# C2DTI Embedding Verification 

In [107]:
# Optional helper: copy embeddings to the preferred location models/embeddings.
from pathlib import Path
import shutil

PROJECT_ROOT = Path.cwd()
source_roots = [
    PROJECT_ROOT / "data-source" / "embeddings",
    PROJECT_ROOT / "data" / "embeddings",
]
destination_root = PROJECT_ROOT / "models" / "embeddings"
destination_root.mkdir(parents=True, exist_ok=True)

copied = []
for root in source_roots:
    if not root.exists():
        continue
    for npz in root.glob("*.npz"):
        dst = destination_root / npz.name
        if not dst.exists():
            shutil.copy2(npz, dst)
            copied.append((npz, dst))

print(f"Destination: {destination_root}")
if copied:
    print(f"Copied {len(copied)} files:")
    for src, dst in copied:
        print(f" - {src} -> {dst}")
else:
    print("No files copied (either none found in source roots, or already present).")

print("Current files in models/embeddings:")
for p in sorted(destination_root.glob("*.npz")):
    print(f" - {p.name}")

Destination: /home/hussen/MINDG/C2DTI/models/embeddings
No files copied (either none found in source roots, or already present).
Current files in models/embeddings:
 - BindingDB_Kd_drug_emb.npz
 - BindingDB_Kd_target_emb.npz
 - DAVIS_ankh_target.npz
 - DAVIS_chemberta_drug.npz
 - DAVIS_drug_emb.npz
 - DAVIS_target_emb.npz
 - KIBA_drug_emb.npz
 - KIBA_target_emb.npz


In [104]:
import numpy as np
import hashlib
from pathlib import Path

print("=" * 80)
print("VERIFYING EMBEDDINGS UNDER models/embeddings (STRICT MODE)")
print("=" * 80)

PROJECT_ROOT = Path.cwd()
emb_root = PROJECT_ROOT / "models" / "embeddings"

datasets = ["DAVIS", "KIBA", "BindingDB_Kd"]
embeddings = ["chemberta", "ankh"]
modalities = {"chemberta": "drug", "ankh": "target"}


def file_hash(path: Path):
    """Compute MD5 hash of file."""
    h = hashlib.md5()
    with path.open("rb") as f:
        h.update(f.read())
    return h.hexdigest()


def candidate_filenames(dataset: str, emb_type: str, modality: str):
    # Support both naming schemes used in this repo.
    names = [f"{dataset}_{modality}_emb.npz"]
    if emb_type == "chemberta" and modality == "drug":
        names.append(f"{dataset}_chemberta_drug.npz")
    if emb_type == "ankh" and modality == "target":
        names.append(f"{dataset}_ankh_target.npz")
    return names


def resolve_file(root: Path, names):
    for name in names:
        p = root / name
        if p.exists():
            return p
    return None


if not emb_root.exists():
    print(f"✗ Missing embedding directory: {emb_root}")
    all_pass = False
else:
    all_pass = True
    for dataset in datasets:
        print(f"\n{'='*80}")
        print(f"Dataset: {dataset}")
        print(f"{'='*80}")

        for emb_type in embeddings:
            modality = modalities[emb_type]
            names = candidate_filenames(dataset, emb_type, modality)
            fpath = resolve_file(emb_root, names)

            if fpath is None:
                print(f"✗ {emb_type.upper()} {modality}: FILE NOT FOUND in {emb_root}")
                print(f"    Tried: {names}")
                all_pass = False
                continue

            try:
                data = np.load(fpath, allow_pickle=True)
                shape = data["embeddings"].shape
                md5 = file_hash(fpath)
                print(f"✓ {emb_type.upper():9} {modality:6} | Shape: {str(shape):15} | md5: {md5[:10]}...")
            except Exception as e:
                print(f"✗ {emb_type.upper()} {modality}: ERROR reading {fpath} - {e}")
                all_pass = False

print(f"\n{'='*80}")
if all_pass:
    print("✓ ALL REQUIRED EMBEDDINGS ARE PRESENT IN models/embeddings")
else:
    print("✗ EMBEDDING CHECK FAILED IN STRICT MODE")
print(f"{'='*80}")

VERIFYING EMBEDDINGS UNDER models/embeddings (STRICT MODE)

Dataset: DAVIS
✓ CHEMBERTA drug   | Shape: (68, 384)       | md5: 9b50cc5cb9...
✓ ANKH      target | Shape: (379, 768)      | md5: 312fd179d1...

Dataset: KIBA
✓ CHEMBERTA drug   | Shape: (462, 384)      | md5: 977b048588...
✓ ANKH      target | Shape: (224, 768)      | md5: f381e5c8b1...

Dataset: BindingDB_Kd
✓ CHEMBERTA drug   | Shape: (10636, 384)    | md5: 32b58a53d4...
✓ ANKH      target | Shape: (1090, 768)     | md5: 268afb1d71...

✓ ALL REQUIRED EMBEDDINGS ARE PRESENT IN models/embeddings


## 4-Pillar Causal Build Notebook

This notebook should lead with the big C2DTI objective, not just repository maintenance.

Primary use:
1. Audit the current implementation against the 4-pillar roadmap.
2. Identify which causal modules already exist in the repo.
3. Expose the missing pieces needed for the journal-grade pipeline.
4. Keep execution cells available as support tools, not as the notebook's main story.

The operational `make` cells below are still useful, but they are secondary. The main focus here is the causal roadmap, implementation status, and the next controlling engineering slice.

In [102]:
from pathlib import Path
import json
import subprocess
from typing import Iterable, Optional

PROJECT_ROOT = Path.cwd()

# This helper runs repository commands from the project root and prints the full result.
def run_repo_command(command: str):
    print(f"$ {command}")
    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        shell=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    print(f"exit_code={completed.returncode}")
    return completed

# This helper reads the newest run summary so notebook users can inspect the latest contract output.
def show_latest_summary(base_dir: Path = PROJECT_ROOT / "outputs" / "runs"):
    if not base_dir.exists():
        print(f"No run directory found at {base_dir}")
        return None

    run_dirs = sorted([path for path in base_dir.iterdir() if path.is_dir()], key=lambda path: path.stat().st_mtime)
    if not run_dirs:
        print(f"No run subdirectories found in {base_dir}")
        return None

    latest_run = run_dirs[-1]
    summary_path = latest_run / "summary.json"
    print(f"latest_run={latest_run}")
    if not summary_path.exists():
        print(f"summary.json not found in {latest_run}")
        return None

    with summary_path.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)

    print(json.dumps(summary, indent=2))
    return summary

# This helper shows the newest files that match a suffix under a directory.
def show_recent_files(base_dir: Path, suffixes: Optional[Iterable[str]] = None, limit: int = 10):
    if not base_dir.exists():
        print(f"No directory found at {base_dir}")
        return []

    files = [path for path in base_dir.rglob("*") if path.is_file()]
    if suffixes is not None:
        suffixes = tuple(suffixes)
        files = [path for path in files if path.name.endswith(suffixes)]

    files = sorted(files, key=lambda path: path.stat().st_mtime, reverse=True)
    selected = files[:limit]
    for path in selected:
        print(path)
    return selected

# This helper reads a text file safely for roadmap auditing.
def read_text_if_exists(path: Path) -> str:
    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8")

# This helper checks whether a file contains all required signal strings.
def file_contains_all(path: Path, needles: Iterable[str]) -> bool:
    text = read_text_if_exists(path)
    return bool(text) and all(needle in text for needle in needles)

# This helper audits the repository against the 4-pillar roadmap in a simple, notebook-friendly way.
def audit_four_pillar_status():
    checks = [
        {
            "phase": "Phase 1",
            "pillar": "Dual pretrained backbone",
            "status_rule": [
                (PROJECT_ROOT / "src/c2dti/backbones.py", ["load_frozen_entity_embeddings"]),
                (PROJECT_ROOT / "src/c2dti/dataset_loader.py", []),
            ],
            "stretch_rule": [
                (PROJECT_ROOT / "src/c2dti/backbones.py", ["SequenceViewEncoder"]),
            ],
        },
        {
            "phase": "Phase 2",
            "pillar": "Cross-view causal agreement",
            "status_rule": [
                (PROJECT_ROOT / "src/c2dti/causal_objective.py", ["compute_cross_view_causal_metrics"]),
                (PROJECT_ROOT / "src/c2dti/perturbation.py", []),
            ],
            "stretch_rule": [
                (PROJECT_ROOT / "src/c2dti/training_loop.py", []),
            ],
        },
        {
            "phase": "Phase 3",
            "pillar": "MAS self-supervision",
            "status_rule": [
                (PROJECT_ROOT / "src/c2dti/backbones.py", ["class MASHead"]),
                (PROJECT_ROOT / "src/c2dti/causal_objective.py", ["compute_mas_losses"]),
            ],
            "stretch_rule": [],
        },
        {
            "phase": "Phase 4",
            "pillar": "IRM plus counterfactual",
            "status_rule": [
                (PROJECT_ROOT / "src/c2dti/irm_loss.py", ["compute_irm_penalty", "compute_counterfactual_loss"]),
                (PROJECT_ROOT / "src/c2dti/causal_objective.py", ["compute_irm_cf_losses"]),
            ],
            "stretch_rule": [],
        },
        {
            "phase": "Phase 5",
            "pillar": "Unified training pipeline",
            "status_rule": [
                (PROJECT_ROOT / "src/c2dti/unified_scorer.py", ["class UnifiedC2DTIScorer"]),
            ],
            "stretch_rule": [
                (PROJECT_ROOT / "src/c2dti/unified_model.py", []),
                (PROJECT_ROOT / "src/c2dti/training.py", []),
            ],
        },
        {
            "phase": "Phase 6",
            "pillar": "Evaluation matrix and ablations",
            "status_rule": [
                (PROJECT_ROOT / "scripts/run_eval_matrix.py", []),
                (PROJECT_ROOT / "scripts/compile_results.py", []),
            ],
            "stretch_rule": [
                (PROJECT_ROOT / "configs/c2dti_full.yaml", []),
                (PROJECT_ROOT / "configs/c2dti_no_causal.yaml", []),
                (PROJECT_ROOT / "configs/c2dti_no_irm.yaml", []),
                (PROJECT_ROOT / "configs/c2dti_no_cf.yaml", []),
                (PROJECT_ROOT / "configs/c2dti_no_mas.yaml", []),
            ],
        },
        {
            "phase": "Phase 7",
            "pillar": "Paper-facing docs",
            "status_rule": [
                (PROJECT_ROOT / "docs/C2DTI_4PILLAR_IMPLEMENTATION_ROADMAP.md", []),
            ],
            "stretch_rule": [
                (PROJECT_ROOT / "docs/results_analysis.md", []),
                (PROJECT_ROOT / "docs/paper_draft.md", []),
            ],
        },
    ]

    audit_rows = []
    for item in checks:
        base_ready = all(
            path.exists() and (not signals or file_contains_all(path, signals))
            for path, signals in item["status_rule"]
        )
        stretch_ready = all(
            path.exists() and (not signals or file_contains_all(path, signals))
            for path, signals in item["stretch_rule"]
        ) if item["stretch_rule"] else False

        if base_ready and stretch_ready:
            status = "advanced"
        elif base_ready:
            status = "partial"
        else:
            status = "missing"

        audit_rows.append({
            "phase": item["phase"],
            "pillar": item["pillar"],
            "status": status,
            "base_ready": base_ready,
            "stretch_ready": stretch_ready,
        })

    return audit_rows

## 4-Pillar Implementation Audit

This section checks the actual repository against the roadmap file.

Status meanings:
- `missing`: the controlling file or core signal is not present.
- `partial`: the pillar exists in some practical form, but the full roadmap target is not complete.
- `advanced`: both the base implementation and the roadmap stretch target are present.

This keeps the notebook grounded in what the repo really has today.

In [74]:
four_pillar_status = audit_four_pillar_status()
for row in four_pillar_status:
    print(f"{row['phase']:8} | {row['pillar']:<32} | status={row['status']:<8} | base={row['base_ready']} | stretch={row['stretch_ready']}")

next_focus_candidates = [row for row in four_pillar_status if row['status'] != 'advanced']
if next_focus_candidates:
    next_focus = next_focus_candidates[0]
    print("\nNext controlling slice to improve:")
    print(next_focus)
else:
    print("\nAll roadmap phases are marked advanced by the current audit rules.")

four_pillar_status

Phase 1  | Dual pretrained backbone         | status=advanced | base=True | stretch=True
Phase 2  | Cross-view causal agreement      | status=partial  | base=True | stretch=False
Phase 3  | MAS self-supervision             | status=partial  | base=True | stretch=False
Phase 4  | IRM plus counterfactual          | status=partial  | base=True | stretch=False
Phase 5  | Unified training pipeline        | status=partial  | base=True | stretch=False
Phase 6  | Evaluation matrix and ablations  | status=partial  | base=True | stretch=False
Phase 7  | Paper-facing docs                | status=partial  | base=True | stretch=False

Next controlling slice to improve:
{'phase': 'Phase 2', 'pillar': 'Cross-view causal agreement', 'status': 'partial', 'base_ready': True, 'stretch_ready': False}


[{'phase': 'Phase 1',
  'pillar': 'Dual pretrained backbone',
  'status': 'advanced',
  'base_ready': True,
  'stretch_ready': True},
 {'phase': 'Phase 2',
  'pillar': 'Cross-view causal agreement',
  'status': 'partial',
  'base_ready': True,
  'stretch_ready': False},
 {'phase': 'Phase 3',
  'pillar': 'MAS self-supervision',
  'status': 'partial',
  'base_ready': True,
  'stretch_ready': False},
 {'phase': 'Phase 4',
  'pillar': 'IRM plus counterfactual',
  'status': 'partial',
  'base_ready': True,
  'stretch_ready': False},
 {'phase': 'Phase 5',
  'pillar': 'Unified training pipeline',
  'status': 'partial',
  'base_ready': True,
  'stretch_ready': False},
 {'phase': 'Phase 6',
  'pillar': 'Evaluation matrix and ablations',
  'status': 'partial',
  'base_ready': True,
  'stretch_ready': False},
 {'phase': 'Phase 7',
  'pillar': 'Paper-facing docs',
  'status': 'partial',
  'base_ready': True,
  'stretch_ready': False}]

## Phase 1 Deep Audit: Dual Pretrained Backbone

This section audits only Phase 1 from the roadmap.

What Phase 1 should deliver:
- frozen pretrained drug and protein backbone support,
- a sequence-view encoder abstraction,
- dataset-side tokenization support for SMILES and protein sequences,
- a clear gap between the current practical implementation and the roadmap target.

The goal here is not to guess. It is to inspect the actual repo and state what already exists, what is missing, and what the next implementation slice should be.

In [75]:
phase1_backbones_text = read_text_if_exists(PROJECT_ROOT / "src/c2dti/backbones.py")
phase1_loader_text = read_text_if_exists(PROJECT_ROOT / "src/c2dti/dataset_loader.py")

phase1_findings = {
    "frozen_embedding_loader": "load_frozen_entity_embeddings" in phase1_backbones_text,
    "sequence_view_encoder": "SequenceViewEncoder" in phase1_backbones_text,
    "mas_head_present": "class MASHead" in phase1_backbones_text,
    "dataset_loader_present": (PROJECT_ROOT / "src/c2dti/dataset_loader.py").exists(),
    "explicit_tokenization_logic": any(
        signal in phase1_loader_text
        for signal in ["token", "tokenizer", "smiles", "sequence-view encoder", "SequenceViewEncoder"]
    ),
}

for key, value in phase1_findings.items():
    print(f"{key:28} -> {value}")

phase1_status = {
    "what_exists_now": [
        "Frozen embedding loading with safe fallback" if phase1_findings["frozen_embedding_loader"] else None,
        "Dataset loader abstraction for benchmark datasets" if phase1_findings["dataset_loader_present"] else None,
        "MAS head groundwork exists in backbones" if phase1_findings["mas_head_present"] else None,
    ],
    "what_is_missing_for_roadmap": [
        "SequenceViewEncoder class" if not phase1_findings["sequence_view_encoder"] else None,
        "Explicit tokenization path for on-the-fly SMILES and protein sequence backbone inputs" if not phase1_findings["explicit_tokenization_logic"] else None,
    ],
}
phase1_status["what_exists_now"] = [item for item in phase1_status["what_exists_now"] if item is not None]
phase1_status["what_is_missing_for_roadmap"] = [item for item in phase1_status["what_is_missing_for_roadmap"] if item is not None]

print("\nWhat exists now:")
for item in phase1_status["what_exists_now"]:
    print(f"- {item}")

print("\nWhat is missing for roadmap-level Phase 1:")
for item in phase1_status["what_is_missing_for_roadmap"]:
    print(f"- {item}")

phase1_next_step = {
    "phase": "Phase 1",
    "next_code_slice": "Add a non-breaking SequenceViewEncoder path in src/c2dti/backbones.py",
    "why": "The repo already has frozen embedding loading, but not the roadmap-level sequence-view abstraction.",
    "follow_up": "After that, extend dataset_loader with explicit tokenization-compatible inputs without breaking current NPZ-based runs.",
}

print("\nRecommended next implementation step:")
for key, value in phase1_next_step.items():
    print(f"{key}: {value}")

phase1_status

frozen_embedding_loader      -> True
sequence_view_encoder        -> True
mas_head_present             -> True
dataset_loader_present       -> True
explicit_tokenization_logic  -> True

What exists now:
- Frozen embedding loading with safe fallback
- Dataset loader abstraction for benchmark datasets
- MAS head groundwork exists in backbones

What is missing for roadmap-level Phase 1:

Recommended next implementation step:
phase: Phase 1
next_code_slice: Add a non-breaking SequenceViewEncoder path in src/c2dti/backbones.py
why: The repo already has frozen embedding loading, but not the roadmap-level sequence-view abstraction.
follow_up: After that, extend dataset_loader with explicit tokenization-compatible inputs without breaking current NPZ-based runs.


{'what_exists_now': ['Frozen embedding loading with safe fallback',
  'Dataset loader abstraction for benchmark datasets',
  'MAS head groundwork exists in backbones'],
 'what_is_missing_for_roadmap': []}

## Full Project Health Analysis (Notebook-First)

Run the next cells in order to get a complete and reproducible project analysis directly from this notebook.

What this section checks:
1. Python syntax sanity for core modules.
2. YAML parse sanity for all configs.
3. Dataset-backed config dry-run health (using `datasets/` CSV sources).
4. Targeted test health for the known failure cluster.

After running all cells, use the final summary output to decide whether the project is ready for full reimplementation steps.

In [76]:
from pathlib import Path
import yaml
import subprocess
import json

PROJECT_ROOT = Path.cwd()

# 1) Core syntax sanity check
syntax_targets = [
    PROJECT_ROOT / "src/c2dti/dataset_loader.py",
    PROJECT_ROOT / "src/c2dti/data_check.py",
    PROJECT_ROOT / "src/c2dti/runner.py",
]

syntax_errors = []
for target in syntax_targets:
    result = subprocess.run(
        ["python", "-m", "py_compile", str(target)],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        syntax_errors.append({
            "file": str(target),
            "stderr": result.stderr.strip(),
        })

# 2) YAML parse sanity for all configs
yaml_errors = []
for cfg in sorted((PROJECT_ROOT / "configs").glob("*.yaml")):
    try:
        yaml.safe_load(cfg.read_text(encoding="utf-8"))
    except Exception as exc:
        yaml_errors.append({"config": str(cfg), "error": str(exc)})

print("syntax_error_count:", len(syntax_errors))
print("yaml_error_count:", len(yaml_errors))

if syntax_errors:
    print("\nSyntax errors:")
    for item in syntax_errors:
        print(item["file"])
        print(item["stderr"])

if yaml_errors:
    print("\nYAML errors:")
    for item in yaml_errors:
        print(item)

health_bootstrap = {
    "syntax_errors": syntax_errors,
    "yaml_errors": yaml_errors,
}
health_bootstrap

syntax_error_count: 0
yaml_error_count: 0


{'syntax_errors': [], 'yaml_errors': []}

In [77]:
# 3) Dataset-backed dry-run health checks for canonical datasets/ configs
dataset_backed_configs = [
    "configs/davis_pipeline.yaml",
    "configs/kiba_pipeline.yaml",
    "configs/bindingdb_pipeline.yaml",
]

dry_run_results = {}
for cfg in dataset_backed_configs:
    completed = subprocess.run(
        ["python", "scripts/run.py", "--config", cfg, "--dry-run"],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )
    dry_run_results[cfg] = {
        "returncode": completed.returncode,
        "stdout_tail": "\n".join(completed.stdout.splitlines()[-8:]),
        "stderr_tail": "\n".join(completed.stderr.splitlines()[-8:]),
    }

for cfg, result in dry_run_results.items():
    print(f"[{cfg}] returncode={result['returncode']}")
    if result["stdout_tail"]:
        print(result["stdout_tail"])
    if result["stderr_tail"]:
        print(result["stderr_tail"])
    print("-" * 80)

dry_run_results

[configs/davis_pipeline.yaml] returncode=0
[OK] Dry-run passed
name=C2DTI_DAVIS_PIPELINE
protocol=P1
output.base_dir=/home/hussen/MINDG/C2DTI/outputs
dataset.name=DAVIS
dataset.path=/home/hussen/MINDG/C2DTI/datasets/DAVIS.csv
model.name=simple_baseline
config=configs/davis_pipeline.yaml
--------------------------------------------------------------------------------
[configs/kiba_pipeline.yaml] returncode=0
[OK] Dry-run passed
name=C2DTI_KIBA_PIPELINE
protocol=P1
output.base_dir=/home/hussen/MINDG/C2DTI/outputs
dataset.name=KIBA
dataset.path=/home/hussen/MINDG/C2DTI/datasets/KIBA.csv
model.name=simple_baseline
config=configs/kiba_pipeline.yaml
--------------------------------------------------------------------------------
[configs/bindingdb_pipeline.yaml] returncode=0
[OK] Dry-run passed
name=C2DTI_BINDINGDB_PIPELINE
protocol=P1
output.base_dir=/home/hussen/MINDG/C2DTI/outputs
dataset.name=BindingDB
dataset.path=/home/hussen/MINDG/C2DTI/datasets/BindingDB_Kd.csv
model.name=simple_base

{'configs/davis_pipeline.yaml': {'returncode': 0,
  'stdout_tail': '[OK] Dry-run passed\nname=C2DTI_DAVIS_PIPELINE\nprotocol=P1\noutput.base_dir=/home/hussen/MINDG/C2DTI/outputs\ndataset.name=DAVIS\ndataset.path=/home/hussen/MINDG/C2DTI/datasets/DAVIS.csv\nmodel.name=simple_baseline\nconfig=configs/davis_pipeline.yaml',
  'stderr_tail': ''},
 'configs/kiba_pipeline.yaml': {'returncode': 0,
  'stdout_tail': '[OK] Dry-run passed\nname=C2DTI_KIBA_PIPELINE\nprotocol=P1\noutput.base_dir=/home/hussen/MINDG/C2DTI/outputs\ndataset.name=KIBA\ndataset.path=/home/hussen/MINDG/C2DTI/datasets/KIBA.csv\nmodel.name=simple_baseline\nconfig=configs/kiba_pipeline.yaml',
  'stderr_tail': ''},
 'configs/bindingdb_pipeline.yaml': {'returncode': 0,
  'stdout_tail': '[OK] Dry-run passed\nname=C2DTI_BINDINGDB_PIPELINE\nprotocol=P1\noutput.base_dir=/home/hussen/MINDG/C2DTI/outputs\ndataset.name=BindingDB\ndataset.path=/home/hussen/MINDG/C2DTI/datasets/BindingDB_Kd.csv\nmodel.name=simple_baseline\nconfig=config

In [78]:
# 4) Targeted test cluster health (the previous failing set)
test_cmd = [
    "python",
    "-m",
    "pytest",
    "tests/test_dataset_loader.py",
    "tests/test_data_check.py",
    "tests/test_splitter.py::TestRunnerSplitIntegration",
    "tests/test_training.py::TestRunnerTrainingIntegration::test_matrix_factorization_run_produces_checkpoint_on_real_dataset",
    "-q",
    "--tb=line",
]

test_run = subprocess.run(
    test_cmd,
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print("targeted_test_returncode:", test_run.returncode)
print("\n=== stdout tail ===")
print("\n".join(test_run.stdout.splitlines()[-40:]))
print("\n=== stderr tail ===")
print("\n".join(test_run.stderr.splitlines()[-20:]))

{"returncode": test_run.returncode}

targeted_test_returncode: 0

=== stdout tail ===
................................                                         [100%]
32 passed in 0.31s

=== stderr tail ===



{'returncode': 0}

In [79]:
# 5) Executive summary for decision-making
syntax_ok = len(health_bootstrap["syntax_errors"]) == 0
yaml_ok = len(health_bootstrap["yaml_errors"]) == 0
dataset_dry_run_ok = all(v["returncode"] == 0 for v in dry_run_results.values())
targeted_tests_ok = (test_run.returncode == 0)

overall_ready = all([syntax_ok, yaml_ok, dataset_dry_run_ok, targeted_tests_ok])

summary = {
    "syntax_ok": syntax_ok,
    "yaml_ok": yaml_ok,
    "dataset_backed_dry_run_ok": dataset_dry_run_ok,
    "targeted_tests_ok": targeted_tests_ok,
    "overall_ready_for_next_reimplementation_step": overall_ready,
}

print(json.dumps(summary, indent=2))
summary

{
  "syntax_ok": true,
  "yaml_ok": true,
  "dataset_backed_dry_run_ok": true,
  "targeted_tests_ok": true,
  "overall_ready_for_next_reimplementation_step": true
}


{'syntax_ok': True,
 'yaml_ok': True,
 'dataset_backed_dry_run_ok': True,
 'targeted_tests_ok': True,
 'overall_ready_for_next_reimplementation_step': True}

## Phase 1 — SequenceViewEncoder (Dual Backbone, Pillar 1)

**Goal:** Add a pure-numpy sequence encoder that converts raw SMILES strings (drugs)
and amino-acid sequences (proteins) into fixed-dimensional feature vectors.

**Why this matters (beginner-friendly):**
- Right now the pipeline only works if pre-computed NPZ embeddings exist on disk.
- `SequenceViewEncoder` gives us a *second view* of each entity using character-level
  n-gram counting — no GPU, no pre-training, no external files needed.
- The two views (frozen embeddings + sequence view) can be compared causally later in
  Pillars 2–4 to check whether each view agrees on which drugs bind which targets.

**Plan in this notebook:**
1. **Prototype** the class here (cell below) — iterate freely.
2. **Quick smoke-test** the prototype on tiny toy data.
3. **Migrate** the final class into `src/c2dti/backbones.py`.
4. **Verify** by importing from source and running the full test suite.


In [80]:
# =============================================================================
# PROTOTYPE — SequenceViewEncoder
# This cell defines the class inline so we can iterate quickly.
# Once tests pass, we copy the final version into src/c2dti/backbones.py.
# =============================================================================

from __future__ import annotations
from collections import Counter
from typing import List, Optional, Sequence
import numpy as np


class SequenceViewEncoder:
    """Character n-gram encoder for drug SMILES and protein amino-acid sequences.

    Beginner-friendly explanation:
    --------------------------------
    A SMILES string like "CC(=O)Nc1ccc(O)cc1" is just text.
    We can turn it into a fixed-size number vector by counting how often each
    short sub-string (n-gram) appears in it.  For example, with n=2:
      "CCN" → bigrams: "CC", "CN"  → count each one.
    We hash every n-gram into a bucket (0 … vocab_size-1) using Python's built-in
    hash function, then count how many n-grams fell into each bucket.
    That count vector is the embedding for this molecule.

    The same idea works for proteins: "MKTAYIAKQR…" is counted with n=3 (trigrams),
    because amino-acid context matters over slightly longer windows.

    Why this is useful:
    - It requires NO pre-training, NO GPU, NO external files.
    - It is deterministic — same string → same vector, every time.
    - It gives a "raw sequence view" that is independent of the frozen NPZ embeddings,
      which is exactly what the causal dual-backbone (Pillar 1) needs.

    Args:
        ngram_n:    Length of character n-grams (default 2 for drugs, 3 for proteins).
        vocab_size: Number of hash buckets = output embedding dimension.
        normalize:  If True, L2-normalize each row so magnitudes are comparable.
    """

    def __init__(
        self,
        ngram_n: int = 2,
        vocab_size: int = 512,
        normalize: bool = True,
    ) -> None:
        self.ngram_n = ngram_n
        self.vocab_size = vocab_size
        self.normalize = normalize

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _ngrams(self, text: str) -> List[str]:
        """Extract all overlapping n-grams from a string.

        Example: _ngrams("ACGT", n=2) → ["AC", "CG", "GT"]
        """
        n = self.ngram_n
        # Pad short strings so every input produces at least one n-gram.
        if len(text) < n:
            text = text.ljust(n, "_")
        return [text[i : i + n] for i in range(len(text) - n + 1)]

    def _encode_one(self, text: str) -> np.ndarray:
        """Encode a single string into a vocab_size-dimensional count vector.

        Step-by-step:
        1. Extract n-grams from the string.
        2. Hash each n-gram into a bucket index (0 … vocab_size-1).
        3. Count how many n-grams landed in each bucket.
        4. (Optionally) L2-normalize the result.
        """
        vec = np.zeros(self.vocab_size, dtype=np.float32)
        for gram in self._ngrams(text):
            # Python's built-in hash is fast but can be negative → take abs + mod.
            bucket = abs(hash(gram)) % self.vocab_size
            vec[bucket] += 1.0
        if self.normalize:
            norm = np.linalg.norm(vec)
            if norm > 0:
                vec /= norm
        return vec

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def encode(self, sequences: Sequence[str]) -> np.ndarray:
        """Encode a list of strings into an (n_sequences, vocab_size) matrix.

        Args:
            sequences: List of SMILES or amino-acid strings.

        Returns:
            Float32 matrix of shape (len(sequences), vocab_size).
        """
        if len(sequences) == 0:
            return np.zeros((0, self.vocab_size), dtype=np.float32)
        return np.stack([self._encode_one(s) for s in sequences], axis=0)

    def __repr__(self) -> str:
        return (
            f"SequenceViewEncoder(ngram_n={self.ngram_n}, "
            f"vocab_size={self.vocab_size}, normalize={self.normalize})"
        )


print("SequenceViewEncoder prototype defined.")
print(repr(SequenceViewEncoder()))


SequenceViewEncoder prototype defined.
SequenceViewEncoder(ngram_n=2, vocab_size=512, normalize=True)


In [81]:
# =============================================================================
# SMOKE TESTS — Validate the prototype before writing to source
# =============================================================================

import numpy as np

# ---------- toy data ----------
TOY_SMILES = [
    "CC(=O)Nc1ccc(O)cc1",   # paracetamol
    "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",  # testosterone
    "c1ccc2ccccc2c1",  # naphthalene
]
TOY_PROTEINS = [
    "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGEDEDTLSLQIRGEALAAISTPVTPPGGELVHPDALDHLNHHVAFSGRSPGRFNNFGLGWCQLGGSKPQDQFQALSARGHEKRHPVSGNRSPGPRGIQGFKIIAGTSPSTPQPASPGSKRGNNPNKKVNGRQTVNGRSHAGVQFTTGHVGELAQESFEPFSVVHAQLPSTAALRLLQDAVDKLPSMWSSPKQAPAADPSSPFLRFPLPHPDEPEQHQTPVDQGQLAAGPAGAP",
    "ACDEFGHIKLMNPQRSTVWY",  # minimal AA alphabet sample
]

drug_enc  = SequenceViewEncoder(ngram_n=2, vocab_size=512, normalize=True)
prot_enc  = SequenceViewEncoder(ngram_n=3, vocab_size=512, normalize=True)

D = drug_enc.encode(TOY_SMILES)
P = prot_enc.encode(TOY_PROTEINS)

# ---------- assertions ----------
assert D.shape == (3, 512), f"Drug matrix shape wrong: {D.shape}"
assert P.shape == (2, 512), f"Protein matrix shape wrong: {P.shape}"
assert D.dtype == np.float32,  "Expected float32"
assert P.dtype == np.float32,  "Expected float32"

# Normalized rows should have L2 norm ≈ 1.0
drug_norms = np.linalg.norm(D, axis=1)
prot_norms = np.linalg.norm(P, axis=1)
assert np.allclose(drug_norms, 1.0, atol=1e-5), f"Drug norms not 1: {drug_norms}"
assert np.allclose(prot_norms, 1.0, atol=1e-5), f"Protein norms not 1: {prot_norms}"

# Same string → same vector (determinism)
D2 = drug_enc.encode(TOY_SMILES)
assert np.array_equal(D, D2), "Not deterministic!"

# Different strings → different vectors (sanity)
assert not np.array_equal(D[0], D[1]), "All drug vectors identical — encoding collapsed!"

# Empty list → (0, vocab_size)
assert drug_enc.encode([]).shape == (0, 512)

# Short string (< ngram_n) handled without crash
enc_short = drug_enc.encode(["C"])
assert enc_short.shape == (1, 512)

# Unnormalized variant
raw_enc = SequenceViewEncoder(ngram_n=2, vocab_size=512, normalize=False)
D_raw = raw_enc.encode(TOY_SMILES)
assert D_raw.sum() > 0, "Raw counts should be non-zero"

print("All smoke tests PASSED ✓")
print(f"  Drug matrix  : {D.shape}  norms={drug_norms.round(4)}")
print(f"  Protein matrix: {P.shape}  norms={prot_norms.round(4)}")
print(f"  Drug[0] non-zero dims: {(D[0] > 0).sum()} / 512")


All smoke tests PASSED ✓
  Drug matrix  : (3, 512)  norms=[1. 1. 1.]
  Protein matrix: (2, 512)  norms=[1. 1.]
  Drug[0] non-zero dims: 13 / 512


### Step 3 — Import from real source & full test suite

The class is now in `src/c2dti/backbones.py`.  
Next cell imports it from there (not the in-notebook prototype) and re-runs all assertions,  
then runs the full test suite to confirm nothing broke.


In [82]:
import subprocess, sys, importlib

# Force a fresh import from the real source file
# (importlib.reload ensures we pick up any changes made after kernel start)
import src.c2dti.backbones as _bb_module
importlib.reload(_bb_module)
from src.c2dti.backbones import SequenceViewEncoder as SequenceViewEncoderFromSource

# ---- Re-run the same smoke tests against the imported class ----
drug_enc_src = SequenceViewEncoderFromSource(ngram_n=2, vocab_size=512, normalize=True)
prot_enc_src = SequenceViewEncoderFromSource(ngram_n=3, vocab_size=512, normalize=True)

D_src = drug_enc_src.encode(TOY_SMILES)
P_src = prot_enc_src.encode(TOY_PROTEINS)

assert D_src.shape == (3, 512)
assert P_src.shape == (2, 512)
assert np.allclose(np.linalg.norm(D_src, axis=1), 1.0, atol=1e-5)
# Verify imported class produces identical results to prototype
assert np.array_equal(D_src, D), "Source import result differs from prototype!"
print(f"Import from source  : OK — {SequenceViewEncoderFromSource.__module__}")
print(f"Results match prototype: {np.array_equal(D_src, D)}")

# ---- Run the full test suite ----
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q", "--tb=short"],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
print("\n--- pytest output ---")
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-1000:])
print(f"\npytest exit code: {result.returncode}")
tests_ok = result.returncode == 0
print(f"\n{'ALL TESTS PASSED ✓' if tests_ok else 'SOME TESTS FAILED ✗'}")


Import from source  : OK — src.c2dti.backbones
Results match prototype: True

--- pytest output ---
........................................................................ [ 47%]
........................................................................ [ 94%]
.........                                                                [100%]
=============================== warnings summary ===============================
tests/test_binary_runner.py::TestBinaryRunner::test_run_once_binary_writes_causal_summary
  /home/hussen/MINDG/C2DTI/src/c2dti/dti_model.py:184: RuntimeWarning: Mean of empty slice
    col_means = np.nanmean(interactions, axis=0, keepdims=True)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
153 passed, 1 warning in 2.69s


pytest exit code: 0

ALL TESTS PASSED ✓


## Architecture Verification — 2 Objectives × 2 Models

This section validates your requested setup explicitly:

1. Objective modes are both wired:
   - binary classification
   - regression
2. Model families are both wired:
   - MixHop (`mixhop_propagation`)
   - Interaction Cross-Attention (`interaction_cross_attention`)

Validation method:
- Run objective-mode unit tests for both models.
- Run config validation tests for `model.objective`.
- Build a compact pass/fail matrix in notebook output.


In [83]:
import subprocess, sys
from src.c2dti.dti_model import create_predictor

checks = {}

# 1) Factory wiring checks
p_mix_reg = create_predictor({"name": "mixhop_propagation", "objective": "regression"})
p_ica_bin = create_predictor({"name": "interaction_cross_attention", "objective": "binary"})
checks["factory_mixhop_objective_regression"] = (getattr(p_mix_reg, "objective", None) == "regression")
checks["factory_interaction_objective_binary"] = (getattr(p_ica_bin, "objective", None) == "binary_classification")

# 2) Run focused architecture tests
cmd = [
    sys.executable,
    "-m",
    "pytest",
    "tests/test_objective_modes.py",
    "tests/test_config_validation.py",
    "-q",
    "--tb=short",
]
res = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)

checks["pytest_objective_and_config_tests"] = (res.returncode == 0)

# 3) Human-readable matrix
matrix = {
    "binary_classification": {
        "mixhop_propagation": checks["factory_mixhop_objective_regression"] is False,  # sanity placeholder for row completeness
        "interaction_cross_attention": checks["factory_interaction_objective_binary"],
    },
    "regression": {
        "mixhop_propagation": checks["factory_mixhop_objective_regression"],
        "interaction_cross_attention": checks["pytest_objective_and_config_tests"],
    },
}

# Fix matrix binary/mixhop entry using direct factory check for binary objective
p_mix_bin = create_predictor({"name": "mixhop_propagation", "objective": "binary_classification"})
matrix["binary_classification"]["mixhop_propagation"] = (getattr(p_mix_bin, "objective", None) == "binary_classification")

architecture_ready = all(
    [
        checks["factory_mixhop_objective_regression"],
        checks["factory_interaction_objective_binary"],
        checks["pytest_objective_and_config_tests"],
        matrix["binary_classification"]["mixhop_propagation"],
        matrix["binary_classification"]["interaction_cross_attention"],
        matrix["regression"]["mixhop_propagation"],
        matrix["regression"]["interaction_cross_attention"],
    ]
)

print("--- Objective/Model Matrix ---")
for obj, row in matrix.items():
    print(f"{obj}: {row}")

print("\n--- Test Output Tail ---")
stdout_tail = res.stdout[-1500:] if len(res.stdout) > 1500 else res.stdout
print(stdout_tail)
if res.returncode != 0:
    stderr_tail = res.stderr[-800:] if len(res.stderr) > 800 else res.stderr
    print(stderr_tail)

print(f"\narchitecture_ready={architecture_ready}")


--- Objective/Model Matrix ---
binary_classification: {'mixhop_propagation': True, 'interaction_cross_attention': True}
regression: {'mixhop_propagation': True, 'interaction_cross_attention': True}

--- Test Output Tail ---
.............                                                            [100%]
13 passed in 0.22s


architecture_ready=True


## Deep Dive — Objective Mode Implementation (Interactive Testing)

Below are isolated code snippets showing exactly how the objective-mode system works.
Execute each cell in order to understand the flow and test behavior.


In [84]:
# ============================================================================
# PART 1: Helper Functions for Objective Detection & Normalization
# ============================================================================
from typing import Optional

def _normalize_objective_name(objective: Optional[str]) -> str:
    """Normalize objective aliases into canonical names.
    
    Canonical names:
      - "auto"                     ← will auto-detect from labels
      - "binary_classification"    ← forces binary mode
      - "regression"               ← forces regression mode
    
    Aliases:
      - "binary" → "binary_classification"
      - "classification" → "binary_classification"
      - "regression" → "regression"
      - "continuous" → "regression"
    """
    if objective is None:
        return "auto"
    
    name = str(objective).strip().lower()
    aliases = {
        "auto": "auto",
        "binary": "binary_classification",
        "binary_classification": "binary_classification",
        "classification": "binary_classification",
        "regression": "regression",
        "continuous": "regression",
    }
    if name not in aliases:
        raise ValueError(
            f"Unsupported objective: {objective!r}. "
            "Must be one of: auto, binary_classification, regression"
        )
    return aliases[name]


def _infer_objective_from_labels(y: np.ndarray, known_mask: np.ndarray) -> str:
    """Infer objective from observed labels.
    
    If all observed labels are in {0, 1}, treat as binary classification.
    Otherwise treat as regression.
    """
    observed = np.asarray(y, dtype=np.float64)[known_mask]
    observed = observed[np.isfinite(observed)]
    if observed.size == 0:
        return "regression"
    
    # Check if all values are 0 or 1 (within tolerance)
    is_binary = np.all(np.isclose(observed, 0.0) | np.isclose(observed, 1.0))
    return "binary_classification" if bool(is_binary) else "regression"


def _resolve_objective(configured_objective: str, y: np.ndarray, known_mask: np.ndarray) -> str:
    """Resolve objective from config and data labels.
    
    If configured_objective == "auto", infer from data.
    Otherwise, trust the config.
    """
    if configured_objective == "auto":
        return _infer_objective_from_labels(y, known_mask)
    return configured_objective


print("✓ Three helper functions defined:")
print("  1. _normalize_objective_name()   — normalize aliases")
print("  2. _infer_objective_from_labels() — auto-detect from data")
print("  3. _resolve_objective()          — resolve config + data")


✓ Three helper functions defined:
  1. _normalize_objective_name()   — normalize aliases
  2. _infer_objective_from_labels() — auto-detect from data
  3. _resolve_objective()          — resolve config + data


In [85]:
# ============================================================================
# PART 1.A: Test Helper Functions with Toy Data
# ============================================================================
import numpy as np

print("=" * 80)
print("TEST 1: Normalize objective name aliases")
print("=" * 80)

test_aliases = [
    ("binary", "binary_classification"),
    ("classification", "binary_classification"),
    ("regression", "regression"),
    ("continuous", "regression"),
    ("auto", "auto"),
    (None, "auto"),
]

for user_input, expected in test_aliases:
    result = _normalize_objective_name(user_input)
    status = "✓" if result == expected else "✗"
    print(f"{status} {str(user_input):20} → {result:25} (expected: {expected})")

print("\n" + "=" * 80)
print("TEST 2: Infer objective from binary labels {0, 1}")
print("=" * 80)

# All labels are 0 or 1 → should be binary
y_binary = np.array([[1, 0, 1], [0, 1, 0]], dtype=np.float32)
known_mask = ~np.isnan(y_binary)
objective = _infer_objective_from_labels(y_binary, known_mask)
print(f"✓ Labels: {y_binary.ravel()}")
print(f"  Inferred objective: {objective} (expected: binary_classification)")

print("\n" + "=" * 80)
print("TEST 3: Infer objective from regression labels (continuous values)")
print("=" * 80)

# Labels have values > 1 → should be regression
y_regression = np.array([[2.3, 3.1, 4.2], [4.8, 5.7, 6.9]], dtype=np.float32)
known_mask = ~np.isnan(y_regression)
objective = _infer_objective_from_labels(y_regression, known_mask)
print(f"✓ Labels: {y_regression.ravel()}")
print(f"  Inferred objective: {objective} (expected: regression)")

print("\n" + "=" * 80)
print("TEST 4: Resolve objective with 'auto' config")
print("=" * 80)

# With auto + binary labels → resolved to binary_classification
y = np.array([[1, 0], [0, 1]], dtype=np.float32)
known_mask = ~np.isnan(y)
resolved = _resolve_objective("auto", y, known_mask)
print(f"✓ Config: 'auto', Data: binary labels {y.ravel()}")
print(f"  Resolved objective: {resolved} (expected: binary_classification)")

# With explicit regression config → use config regardless of data
resolved = _resolve_objective("regression", y, known_mask)
print(f"✓ Config: 'regression', Data: binary labels {y.ravel()}")
print(f"  Resolved objective: {resolved} (expected: regression) ← trusts config, not data")


TEST 1: Normalize objective name aliases
✓ binary               → binary_classification     (expected: binary_classification)
✓ classification       → binary_classification     (expected: binary_classification)
✓ regression           → regression                (expected: regression)
✓ continuous           → regression                (expected: regression)
✓ auto                 → auto                      (expected: auto)
✓ None                 → auto                      (expected: auto)

TEST 2: Infer objective from binary labels {0, 1}
✓ Labels: [1. 0. 1. 0. 1. 0.]
  Inferred objective: binary_classification (expected: binary_classification)

TEST 3: Infer objective from regression labels (continuous values)
✓ Labels: [2.3 3.1 4.2 4.8 5.7 6.9]
  Inferred objective: regression (expected: regression)

TEST 4: Resolve objective with 'auto' config
✓ Config: 'auto', Data: binary labels [1. 0. 0. 1.]
  Resolved objective: binary_classification (expected: binary_classification)
✓ Config: 

In [86]:
# ============================================================================
# PART 2: MixHop Objective-Aware Output Logic
# ============================================================================

print("=" * 80)
print("CONCEPT: How MixHop respects objective mode")
print("=" * 80)

def mixhop_output_logic(raw_pred: np.ndarray, objective: str) -> np.ndarray:
    """Simplified MixHop output logic showing objective-aware clipping.
    
    Args:
        raw_pred: Raw prediction matrix (could have values outside [0, 1])
        objective: Either "binary_classification" or "regression"
    
    Returns:
        Predictions adjusted based on objective mode
    """
    # Binary classification: force predictions to [0, 1] probability range
    if objective == "binary_classification":
        return np.clip(raw_pred, 0.0, 1.0).astype(np.float32)
    
    # Regression: preserve raw values (could be > 1 or < 0)
    return raw_pred.astype(np.float32)


# Test with raw predictions that include values > 1
raw_predictions = np.array([
    [2.3, 0.5, -0.2],
    [0.8, 3.1, 1.5],
], dtype=np.float64)

print("\nRaw predictions (before objective-aware adjustment):")
print(raw_predictions)

# Binary mode: clips to [0, 1]
binary_output = mixhop_output_logic(raw_predictions, "binary_classification")
print("\nBinary classification mode:")
print("  → Clipped to [0, 1]:")
print(binary_output)
print(f"  Max value: {binary_output.max():.2f} (forced ≤ 1.0)")

# Regression mode: preserves all values
regression_output = mixhop_output_logic(raw_predictions, "regression")
print("\nRegression mode:")
print("  → NO clipping (preserves raw values):")
print(regression_output)
print(f"  Max value: {regression_output.max():.2f} (raw value preserved!)")

print("\n✓ Key insight:")
print("  - Binary:    predictions are probabilities in [0, 1]")
print("  - Regression: predictions can be any real number (e.g., 2.3, 3.1, -0.2)")


CONCEPT: How MixHop respects objective mode

Raw predictions (before objective-aware adjustment):
[[ 2.3  0.5 -0.2]
 [ 0.8  3.1  1.5]]

Binary classification mode:
  → Clipped to [0, 1]:
[[1.  0.5 0. ]
 [0.8 1.  1. ]]
  Max value: 1.00 (forced ≤ 1.0)

Regression mode:
  → NO clipping (preserves raw values):
[[ 2.3  0.5 -0.2]
 [ 0.8  3.1  1.5]]
  Max value: 3.10 (raw value preserved!)

✓ Key insight:
  - Binary:    predictions are probabilities in [0, 1]
  - Regression: predictions can be any real number (e.g., 2.3, 3.1, -0.2)


In [87]:
# ============================================================================
# PART 3: InteractionCrossAttention Objective-Aware Output Logic
# ============================================================================

print("=" * 80)
print("CONCEPT: InteractionCrossAttention uses the same objective-aware logic")
print("=" * 80)

def interaction_cross_attention_output_logic(
    mf_score: np.ndarray,
    attn: np.ndarray,
    beta: np.ndarray,
    objective: str,
) -> np.ndarray:
    """Simplified InteractionCrossAttention output logic.
    
    Fuses matrix-factorization score and attention score, then applies
    objective-aware output handling.
    
    Args:
        mf_score: Matrix factorization predictions (n_drugs, n_targets)
        attn: Attention scores (n_drugs, n_targets)
        beta: Calibration weights [intercept, w_mf, w_attn]
        objective: "binary_classification" or "regression"
    
    Returns:
        Final predictions adjusted based on objective
    """
    # Fuse two branches
    pred = beta[0] + beta[1] * mf_score + beta[2] * attn
    
    # Apply objective-aware output
    if objective == "binary_classification":
        pred = np.clip(pred, 0.0, 1.0)
    
    return pred.astype(np.float32)


# Toy data
mf_score = np.array([[2.5, 0.3], [0.8, 3.2]], dtype=np.float64)
attn = np.array([[0.2, 1.8], [1.5, 0.5]], dtype=np.float64)
beta = np.array([0.1, 0.5, 0.5], dtype=np.float64)  # intercept + two weights

print("\nInput components:")
print(f"MF score:\n{mf_score}")
print(f"Attention:\n{attn}")
print(f"Beta (calibration): {beta}")

# Binary mode
pred_binary = interaction_cross_attention_output_logic(mf_score, attn, beta, "binary_classification")
print(f"\nBinary classification mode:")
print(f"  Fused & clipped to [0, 1]:\n{pred_binary}")
print(f"  Value range: [{pred_binary.min():.2f}, {pred_binary.max():.2f}]")

# Regression mode
pred_regression = interaction_cross_attention_output_logic(mf_score, attn, beta, "regression")
print(f"\nRegression mode:")
print(f"  Fused (NO clipping):\n{pred_regression}")
print(f"  Value range: [{pred_regression.min():.2f}, {pred_regression.max():.2f}]")
print(f"  → Some values are > 1.0 (preserved for regression)")


CONCEPT: InteractionCrossAttention uses the same objective-aware logic

Input components:
MF score:
[[2.5 0.3]
 [0.8 3.2]]
Attention:
[[0.2 1.8]
 [1.5 0.5]]
Beta (calibration): [0.1 0.5 0.5]

Binary classification mode:
  Fused & clipped to [0, 1]:
[[1. 1.]
 [1. 1.]]
  Value range: [1.00, 1.00]

Regression mode:
  Fused (NO clipping):
[[1.45 1.15]
 [1.25 1.95]]
  Value range: [1.15, 1.95]
  → Some values are > 1.0 (preserved for regression)


In [88]:
# ============================================================================
# PART 4: Factory Wiring — How YAML config reaches the model
# ============================================================================

print("=" * 80)
print("CONCEPT: Factory reads YAML config and wires objective to model")
print("=" * 80)

def create_mixhop_simplified(model_cfg: dict):
    """Simplified factory for MixHop.
    
    Reads config dict and passes all parameters (including objective)
    to the model constructor.
    """
    class SimplifiedMixHop:
        def __init__(self, objective="auto"):
            self.objective = _normalize_objective_name(objective)
        
        def __repr__(self):
            return f"MixHop(objective={self.objective!r})"
    
    return SimplifiedMixHop(
        objective=str(model_cfg.get("objective", "auto"))
    )


def create_interaction_cross_attention_simplified(model_cfg: dict):
    """Simplified factory for InteractionCrossAttention."""
    class SimplifiedICA:
        def __init__(self, objective="auto"):
            self.objective = _normalize_objective_name(objective)
        
        def __repr__(self):
            return f"InteractionCrossAttention(objective={self.objective!r})"
    
    return SimplifiedICA(
        objective=str(model_cfg.get("objective", "auto"))
    )


# Simulate different YAML configs
configs = [
    {
        "name": "MixHop + Binary",
        "model": {"name": "mixhop_propagation", "objective": "binary_classification"},
    },
    {
        "name": "MixHop + Regression (alias)",
        "model": {"name": "mixhop_propagation", "objective": "continuous"},
    },
    {
        "name": "InteractionCrossAttention + Auto",
        "model": {"name": "interaction_cross_attention", "objective": "auto"},
    },
]

print("\nFactory wiring examples:\n")

for cfg in configs:
    model_cfg = cfg["model"]
    model_name = model_cfg.get("name")
    
    if "mixhop" in model_name:
        predictor = create_mixhop_simplified(model_cfg)
    else:
        predictor = create_interaction_cross_attention_simplified(model_cfg)
    
    print(f"{cfg['name']:40}")
    print(f"  Config: {model_cfg}")
    print(f"  Result: {predictor}")
    print()

print("✓ Factory correctly:")
print("  1. Reads model.objective from YAML config")
print("  2. Passes it to model constructor")
print("  3. Model normalizes aliases (e.g., 'continuous' → 'regression')")


CONCEPT: Factory reads YAML config and wires objective to model

Factory wiring examples:

MixHop + Binary                         
  Config: {'name': 'mixhop_propagation', 'objective': 'binary_classification'}
  Result: MixHop(objective='binary_classification')

MixHop + Regression (alias)             
  Config: {'name': 'mixhop_propagation', 'objective': 'continuous'}
  Result: MixHop(objective='regression')

InteractionCrossAttention + Auto        
  Config: {'name': 'interaction_cross_attention', 'objective': 'auto'}
  Result: InteractionCrossAttention(objective='auto')

✓ Factory correctly:
  1. Reads model.objective from YAML config
  2. Passes it to model constructor
  3. Model normalizes aliases (e.g., 'continuous' → 'regression')


In [89]:
# ============================================================================
# PART 5: Config Validation — Catching Invalid Objectives at Load Time
# ============================================================================

print("=" * 80)
print("CONCEPT: Config validator catches invalid objectives before model init")
print("=" * 80)

def validate_model_objective(objective_value):
    """Simplified config validator for model.objective field."""
    valid = {"auto", "binary", "binary_classification", "classification", "regression", "continuous"}
    normalized = str(objective_value).strip().lower()
    
    if normalized not in valid:
        raise ValueError(
            f"model.objective must be one of: "
            f"{', '.join(sorted(valid))}\n"
            f"Got: {objective_value!r}"
        )
    return True


print("\nValid objectives (all accepted by validator):\n")
valid_objectives = ["auto", "binary", "classification", "binary_classification", "regression", "continuous"]
for obj in valid_objectives:
    try:
        validate_model_objective(obj)
        print(f"  ✓ {obj:25} → accepted")
    except ValueError as e:
        print(f"  ✗ {obj:25} → {e}")

print("\n" + "-" * 80)
print("\nInvalid objectives (rejected by validator):\n")
invalid_objectives = ["probabilistic", "multinomial", "ordinal", "ranking"]
for obj in invalid_objectives:
    try:
        validate_model_objective(obj)
        print(f"  ✓ {obj:25} → accepted")
    except ValueError as e:
        print(f"  ✗ {obj:25} → rejected")
        print(f"     Reason: {str(e).split(chr(10))[0]}")

print("\n✓ Validation ensures:")
print("  1. Only valid objectives reach the model")
print("  2. Typos are caught early (at config load time)")
print("  3. Clear error messages guide users")


CONCEPT: Config validator catches invalid objectives before model init

Valid objectives (all accepted by validator):

  ✓ auto                      → accepted
  ✓ binary                    → accepted
  ✓ classification            → accepted
  ✓ binary_classification     → accepted
  ✓ regression                → accepted
  ✓ continuous                → accepted

--------------------------------------------------------------------------------

Invalid objectives (rejected by validator):

  ✗ probabilistic             → rejected
     Reason: model.objective must be one of: auto, binary, binary_classification, classification, continuous, regression
  ✗ multinomial               → rejected
     Reason: model.objective must be one of: auto, binary, binary_classification, classification, continuous, regression
  ✗ ordinal                   → rejected
     Reason: model.objective must be one of: auto, binary, binary_classification, classification, continuous, regression
  ✗ ranking          

In [65]:
# ============================================================================
# PART 6: End-to-End Flow — Config → Model → Objective-Aware Predictions
# ============================================================================

print("=" * 80)
print("END-TO-END EXAMPLE: YAML config through to objective-aware predictions")
print("=" * 80)

# Scenario 1: Binary classification with DAVIS
print("\n" + "=" * 80)
print("Scenario 1: DAVIS binary classification")
print("=" * 80)

config_binary = {
    "dataset": "DAVIS",
    "model": {
        "name": "mixhop_propagation",
        "objective": "binary",  # alias for binary_classification
    }
}

print(f"\n1. YAML config loaded:")
print(f"   model.objective = {config_binary['model']['objective']!r}")

# Validation step
validate_model_objective(config_binary["model"]["objective"])
print(f"   ✓ Validator passed")

# Factory creates model with objective
model = create_mixhop_simplified(config_binary["model"])
print(f"\n2. Factory creates model:")
print(f"   {model}")

# Now simulate a prediction scenario
print(f"\n3. During prediction:")
# Imagine raw predictions from model internals
raw_pred = np.array([[2.5, -0.1], [0.7, 1.3]], dtype=np.float64)
print(f"   Raw model output: {raw_pred.ravel()}")
print(f"   Objective mode: {model.objective}")
final_pred = mixhop_output_logic(raw_pred, model.objective)
print(f"   Final output (clipped): {final_pred.ravel()}")
print(f"   → All values in [0, 1] (valid probabilities) ✓")

# Scenario 2: Regression with KIBA
print("\n" + "=" * 80)
print("Scenario 2: KIBA regression with InteractionCrossAttention")
print("=" * 80)

config_regression = {
    "dataset": "KIBA",
    "model": {
        "name": "interaction_cross_attention",
        "objective": "continuous",  # alias for regression
    }
}

print(f"\n1. YAML config loaded:")
print(f"   model.objective = {config_regression['model']['objective']!r}")

# Validation step
validate_model_objective(config_regression["model"]["objective"])
print(f"   ✓ Validator passed")

# Factory creates model with objective
model = create_interaction_cross_attention_simplified(config_regression["model"])
print(f"\n2. Factory creates model:")
print(f"   {model}")

# Prediction scenario
print(f"\n3. During prediction:")
mf_score = np.array([[2.1, 0.2], [3.5, 0.9]], dtype=np.float64)
attn = np.array([[0.5, 1.2], [0.3, 2.0]], dtype=np.float64)
beta = np.array([0.05, 0.6, 0.4], dtype=np.float64)
print(f"   Raw model output (MF + attn): min={mf_score.min():.1f}...max={mf_score.max():.1f}")
print(f"   Objective mode: {model.objective}")
final_pred = interaction_cross_attention_output_logic(mf_score, attn, beta, model.objective)
print(f"   Final output (NO clipping): min={final_pred.min():.2f}...max={final_pred.max():.2f}")
print(f"   → Raw values preserved (valid for regression) ✓")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("""
The objective-mode system flows like this:

  YAML Config
      ↓
  [Validation] — Checks: objective ∈ {auto, binary, ..., regression}
      ↓
  [Factory] — Reads model.objective, passes to constructor
      ↓
  [Model Stores] — Normalizes aliases (binary → binary_classification)
      ↓
  [Fit/Predict] — Resolves objective (auto-detect if "auto")
      ↓
  [Output Logic] — Applies clipping only if binary_classification
      ↓
  Final Predictions ∈ [0,1] for binary, ℝ for regression

This ensures:
  ✓ Users can configure objective per model
  ✓ Auto-detection works if not specified
  ✓ Binary and regression modes produce correct output ranges
  ✓ Config errors caught early, not at prediction time
""")


END-TO-END EXAMPLE: YAML config through to objective-aware predictions

Scenario 1: DAVIS binary classification

1. YAML config loaded:
   model.objective = 'binary'
   ✓ Validator passed

2. Factory creates model:
   MixHop(objective='binary_classification')

3. During prediction:
   Raw model output: [ 2.5 -0.1  0.7  1.3]
   Objective mode: binary_classification
   Final output (clipped): [1.  0.  0.7 1. ]
   → All values in [0, 1] (valid probabilities) ✓

Scenario 2: KIBA regression with InteractionCrossAttention

1. YAML config loaded:
   model.objective = 'continuous'
   ✓ Validator passed

2. Factory creates model:
   InteractionCrossAttention(objective='regression')

3. During prediction:
   Raw model output (MF + attn): min=0.2...max=3.5
   Objective mode: regression
   Final output (NO clipping): min=0.65...max=2.27
   → Raw values preserved (valid for regression) ✓

SUMMARY

The objective-mode system flows like this:

  YAML Config
      ↓
  [Validation] — Checks: objective ∈

In [90]:
# ============================================================================
# PART 7: BindingDB_Kd End-to-End (Regression-First)
# ============================================================================

print("=" * 80)
print("Scenario 3: BindingDB_Kd regression")
print("=" * 80)

# NOTE:
# - Correct dataset key in this repo is "BindingDB_Kd" (not "bingdb_kb").
# - This dataset is naturally regression-oriented (continuous affinity values).

config_bindingdb = {
    "dataset": "BindingDB_Kd",
    "model": {
        "name": "mixhop_propagation",
        "objective": "regression",
    },
}

print("\n1. YAML-like config:")
print(config_bindingdb)

validate_model_objective(config_bindingdb["model"]["objective"])
print("2. Objective validation: PASSED")

bindingdb_model = create_mixhop_simplified(config_bindingdb["model"])
print(f"3. Model after factory wiring: {bindingdb_model}")

# Simulate continuous affinity predictions to show no clipping in regression mode
bindingdb_raw_pred = np.array([
    [0.7, 1.9, 3.4],
    [2.2, 4.8, 0.5],
], dtype=np.float64)

bindingdb_final_pred = mixhop_output_logic(bindingdb_raw_pred, bindingdb_model.objective)
print("4. Regression output check (no clipping expected)")
print(f"   raw max:   {bindingdb_raw_pred.max():.2f}")
print(f"   final max: {bindingdb_final_pred.max():.2f}")
print(f"   no_clipping: {bool(bindingdb_final_pred.max() > 1.0)}")

print("\n✓ BindingDB_Kd is correctly handled in regression objective mode.")


Scenario 3: BindingDB_Kd regression

1. YAML-like config:
{'dataset': 'BindingDB_Kd', 'model': {'name': 'mixhop_propagation', 'objective': 'regression'}}
2. Objective validation: PASSED
3. Model after factory wiring: MixHop(objective='regression')
4. Regression output check (no clipping expected)
   raw max:   4.80
   final max: 4.80
   no_clipping: True

✓ BindingDB_Kd is correctly handled in regression objective mode.


In [91]:
# ============================================================================
# PART 8: BindingDB_Kd Binary Counterpart (A/B Comparison)
# ============================================================================

print("=" * 80)
print("Scenario 4: BindingDB_Kd binary counterpart")
print("=" * 80)

config_bindingdb_binary = {
    "dataset": "BindingDB_Kd",
    "model": {
        "name": "mixhop_propagation",
        "objective": "binary_classification",
    },
}

print("\n1. YAML-like config:")
print(config_bindingdb_binary)

validate_model_objective(config_bindingdb_binary["model"]["objective"])
print("2. Objective validation: PASSED")

bindingdb_binary_model = create_mixhop_simplified(config_bindingdb_binary["model"])
print(f"3. Model after factory wiring: {bindingdb_binary_model}")

# Use the SAME raw values as regression cell to show exact behavioral difference
bindingdb_binary_raw_pred = bindingdb_raw_pred.copy()
bindingdb_binary_final_pred = mixhop_output_logic(
    bindingdb_binary_raw_pred,
    bindingdb_binary_model.objective,
)

print("4. Binary output check (clipping expected)")
print(f"   raw max:   {bindingdb_binary_raw_pred.max():.2f}")
print(f"   final max: {bindingdb_binary_final_pred.max():.2f}")
print(f"   clipped_to_unit_interval: {bool(bindingdb_binary_final_pred.max() <= 1.0)}")

print("\n5. A/B comparison on same raw predictions")
print(f"   regression final max: {bindingdb_final_pred.max():.2f}")
print(f"   binary final max:     {bindingdb_binary_final_pred.max():.2f}")
print("   conclusion: regression preserves raw scale, binary enforces probability range [0, 1]")

print("\n✓ BindingDB_Kd binary counterpart behaves as expected.")


Scenario 4: BindingDB_Kd binary counterpart

1. YAML-like config:
{'dataset': 'BindingDB_Kd', 'model': {'name': 'mixhop_propagation', 'objective': 'binary_classification'}}
2. Objective validation: PASSED
3. Model after factory wiring: MixHop(objective='binary_classification')
4. Binary output check (clipping expected)
   raw max:   4.80
   final max: 1.00
   clipped_to_unit_interval: True

5. A/B comparison on same raw predictions
   regression final max: 4.80
   binary final max:     1.00
   conclusion: regression preserves raw scale, binary enforces probability range [0, 1]

✓ BindingDB_Kd binary counterpart behaves as expected.


## Backbones.py Breakout (Step-by-Step)

This section explains and tests the three main components in `src/c2dti/backbones.py`:

1. `load_frozen_entity_embeddings` (frozen NPZ loading + fallback)
2. `MASHead` (masked reconstruction self-supervision)
3. `SequenceViewEncoder` (character n-gram sequence view)

Run cells in order.


In [129]:
from pathlib import Path
import inspect
import numpy as np

# AIM: Confirm the notebook is using the real backbone implementation.
# OBJECTIVE: Import backbone symbols from source and verify their callable signatures.
# INPUT: Source module path and backbone symbol names.
# OUTPUT: Printed symbol names and signatures.
# REAL-DATA NOTE: This confirms your runtime is bound to real project code in src/c2dti/backbones.py.

from src.c2dti.backbones import (
    load_frozen_entity_embeddings,
    MASHead,
    SequenceViewEncoder,
)

PROJECT_ROOT = Path.cwd()

print("Loaded symbols from src.c2dti.backbones")
print("-", load_frozen_entity_embeddings.__name__, inspect.signature(load_frozen_entity_embeddings))
print("-", MASHead.__name__, inspect.signature(MASHead))
print("-", SequenceViewEncoder.__name__, inspect.signature(SequenceViewEncoder))


Loaded symbols from src.c2dti.backbones
- load_frozen_entity_embeddings (entities: 'Sequence[str]', npz_path: 'Optional[str]', default_dim: 'int' = 768, embeddings_key: 'str' = 'embeddings', keys_key: 'str' = 'keys') -> 'np.ndarray'
- MASHead (mask_ratio: 'float' = 0.15, seed: 'int' = 42) -> 'None'
- SequenceViewEncoder (ngram_n: 'int' = 2, vocab_size: 'int' = 512, normalize: 'bool' = True) -> 'None'


In [130]:
# AIM: Validate frozen embedding loading behavior for the real pipeline contract.
# OBJECTIVE: Check file presence, loader output shape/dtype, and fallback path messaging.
# INPUT: entities list, expected NPZ path, and a guaranteed-missing NPZ path.
# OUTPUT: X_real/X_missing shape and dtype, plus fallback non-zero check.
# REAL-DATA NOTE: For final training/evaluation, use real dataset entity IDs in `entities`.

entities = ["drug_a", "drug_b", "drug_c"]
real_npz = PROJECT_ROOT / "models" / "embeddings" / "DAVIS_drug_emb.npz"
missing_npz = PROJECT_ROOT / "models" / "embeddings" / "NOT_EXIST.npz"

print("Real file exists:", real_npz.exists())
print("Missing file exists:", missing_npz.exists())

# Real file path: may or may not align by count/keys, but function should return safely.
X_real = load_frozen_entity_embeddings(
    entities=entities,
    npz_path=str(real_npz),
    default_dim=384,
)
print("X_real shape:", X_real.shape)
print("X_real dtype:", X_real.dtype)

# Missing file path: guaranteed hash fallback.
X_missing = load_frozen_entity_embeddings(
    entities=entities,
    npz_path=str(missing_npz),
    default_dim=384,
)
print("X_missing shape:", X_missing.shape)
print("X_missing dtype:", X_missing.dtype)
print("Fallback produced non-zero rows:", bool(np.any(np.abs(X_missing) > 0)))


Real file exists: True
Missing file exists: False
[Backbone] Incompatible NPZ (/home/hussen/MINDG/C2DTI/models/embeddings/DAVIS_drug_emb.npz): Embedding rows (68) do not match entity count (3) and no 'keys' present. Using hash fallback.
X_real shape: (3, 384)
X_real dtype: float32
[Backbone] Missing NPZ file: /home/hussen/MINDG/C2DTI/models/embeddings/NOT_EXIST.npz. Using hash fallback.
X_missing shape: (3, 384)
X_missing dtype: float32
Fallback produced non-zero rows: True


In [132]:
# AIM: Verify MASHead reconstruction behavior.
# OBJECTIVE: Fit MASHead and compare reconstruction loss on baseline vs shifted inputs.
# INPUT: Embedding matrix (n_samples, d), mask_ratio, seed.
# OUTPUT: Reconstruction losses for baseline and shifted embeddings.
# REAL-DATA NOTE: Replace `emb_toy` with real embedding batches to profile production behavior.

rng = np.random.default_rng(0)
emb_toy = rng.normal(size=(20, 64)).astype(np.float32)

mas = MASHead(mask_ratio=0.2, seed=123)
mas.fit(emb_toy)
loss_same = mas.reconstruct_loss(emb_toy)

# Evaluate on a slightly shifted copy to inspect sensitivity.
emb_shifted = emb_toy + 0.05
loss_shifted = mas.reconstruct_loss(emb_shifted)

print("MASHead fitted")
print("- input shape:", emb_toy.shape)
print("- reconstruction loss (train-like):", float(loss_same))
print("- reconstruction loss (shifted):", float(loss_shifted))


MASHead fitted
- input shape: (20, 64)
- reconstruction loss (train-like): 1.4368041244413425e-30
- reconstruction loss (shifted): 0.00349330551415152


In [133]:
# AIM: Validate sequence-view feature encoding for drug and target inputs.
# OBJECTIVE: Generate fixed-width embeddings and verify normalization and determinism.
# INPUT: Sequence lists, ngram_n, vocab_size, normalize flag.
# OUTPUT: Encoded matrix shapes, row norms, and deterministic equality check.
# REAL-DATA NOTE: Replace sample sequences with real SMILES/protein strings from your dataset.

toy_smiles = ["CCO", "c1ccccc1", "CC(=O)O"]
toy_proteins = ["MKTAYIAK", "ACDEFGHIKLMNP"]

enc_drug = SequenceViewEncoder(ngram_n=2, vocab_size=128, normalize=True)
enc_prot = SequenceViewEncoder(ngram_n=3, vocab_size=128, normalize=True)

Z_drug = enc_drug.encode(toy_smiles)
Z_prot = enc_prot.encode(toy_proteins)

print("Drug embedding matrix:", Z_drug.shape)
print("Protein embedding matrix:", Z_prot.shape)
print("Drug norms:", np.linalg.norm(Z_drug, axis=1).round(4))
print("Protein norms:", np.linalg.norm(Z_prot, axis=1).round(4))
print("Deterministic check (same input -> same output):", np.array_equal(Z_drug, enc_drug.encode(toy_smiles)))


Drug embedding matrix: (3, 128)
Protein embedding matrix: (2, 128)
Drug norms: [1. 1. 1.]
Protein norms: [1. 1.]
Deterministic check (same input -> same output): True


## Causal Objective Breakout (Step-by-Step)

This section explains and tests the causal objective utilities in src/c2dti/causal_objective.py:

1. validate_causal_config (configuration validation)
2. compute_cross_view_causal_metrics (Pillar 2)
3. compute_mas_losses (Pillar 3)
4. compute_irm_cf_losses (Pillar 4)

Run cells in order.

In [134]:
import inspect
import numpy as np

from src.c2dti.causal_objective import (
    validate_causal_config,
    compute_causal_reliability_score,
    compute_cross_view_causal_metrics,
    compute_mas_losses,
    compute_irm_cf_losses,
)

# AIM: Confirm the notebook is using the real causal-objective implementation.
# OBJECTIVE: Import causal utilities from source and verify callable signatures.
# INPUT: Source module path and causal utility symbol names.
# OUTPUT: Printed symbol names and signatures.
# REAL-DATA NOTE: This confirms runtime binding to src/c2dti/causal_objective.py.

print("Loaded symbols from src.c2dti.causal_objective")
print("-", validate_causal_config.__name__, inspect.signature(validate_causal_config))
print("-", compute_causal_reliability_score.__name__, inspect.signature(compute_causal_reliability_score))
print("-", compute_cross_view_causal_metrics.__name__, inspect.signature(compute_cross_view_causal_metrics))
print("-", compute_mas_losses.__name__, inspect.signature(compute_mas_losses))
print("-", compute_irm_cf_losses.__name__, inspect.signature(compute_irm_cf_losses))

Loaded symbols from src.c2dti.causal_objective
- validate_causal_config (causal_cfg: Union[Dict[str, Any], NoneType]) -> list
- compute_causal_reliability_score (baseline_predictions: numpy.ndarray, perturbed_predictions: numpy.ndarray, weight: float = 1.0) -> float
- compute_cross_view_causal_metrics (p_seq: numpy.ndarray, p_graph: numpy.ndarray, p_seq_pert: numpy.ndarray, p_graph_pert: numpy.ndarray, weight: float = 1.0) -> Dict[str, float]
- compute_mas_losses (drug_embeddings: numpy.ndarray, prot_embeddings: numpy.ndarray, mask_ratio: float = 0.15, seed: int = 42, weight: float = 1.0) -> Dict[str, float]
- compute_irm_cf_losses (predictions: numpy.ndarray, labels: numpy.ndarray, n_drugs: int, n_targets: int, n_envs: int = 4, pos_threshold: float = 0.5, n_cf_pairs: int = 1000, irm_weight: float = 1.0, cf_weight: float = 1.0, seed: int = 42) -> Dict[str, float]


In [135]:
# AIM: Validate causal config schema and accepted values.
# OBJECTIVE: Check that valid config passes and invalid config returns clear errors.
# INPUT: Causal config dictionaries.
# OUTPUT: Error list for each config.
# REAL-DATA NOTE: Use this before launching real training runs.

good_cfg = {
    "enabled": True,
    "weight": 0.5,
    "mode": "cross_view",
    "sequence_model": {"name": "sequence_view"},
    "graph_model": {"name": "mixhop"},
}

bad_cfg = {
    "enabled": "yes",
    "weight": -1,
    "mode": "unknown",
    "sequence_model": "not-a-mapping",
}

print("good_cfg errors:", validate_causal_config(good_cfg))
print("bad_cfg errors:", validate_causal_config(bad_cfg))

good_cfg errors: []
bad_cfg errors: ['causal.enabled must be a boolean', 'causal.weight must be a non-negative number', 'causal.mode must be one of: reliability, cross_view, mas, irm_cf, unified', 'causal.sequence_model must be a mapping when provided']


In [138]:
# AIM: Verify Pillar 2 cross-view causal metrics behavior.
# OBJECTIVE: Measure view agreement and perturbed-view consistency.
# INPUT: Baseline and perturbed prediction matrices from sequence and graph views.
# OUTPUT: mse terms, l_xview, weighted loss, bounded causal_score.
# REAL-DATA NOTE: Replace demo matrices with model outputs from real batches.

p_seq = np.array([[0.8, 0.2], [0.6, 0.4]], dtype=np.float32)
p_graph = np.array([[0.75, 0.25], [0.62, 0.38]], dtype=np.float32)
p_seq_pert = np.array([[0.7, 0.3], [0.55, 0.45]], dtype=np.float32)
p_graph_pert = np.array([[0.72, 0.28], [0.58, 0.42]], dtype=np.float32)

xview = compute_cross_view_causal_metrics(
    p_seq=p_seq,
    p_graph=p_graph,
    p_seq_pert=p_seq_pert,
    p_graph_pert=p_graph_pert,
    weight=1.0,
)

print("Cross-view metrics:")
for k, v in xview.items():
    print(f"- {k}: {v}")

Cross-view metrics:
- mse_seq_graph: 0.0014500000979751348
- mse_seqpert_graph: 0.0037000002339482307
- mse_seq_graphpert: 0.0033999993465840816
- l_xview: 0.008549999678507447
- l_xview_weighted: 0.008549999678507447
- causal_score: 0.9915224830883616


In [137]:
# AIM: Verify Pillar 3 (MAS) and Pillar 4 (IRM+CF) objective terms run end-to-end.
# OBJECTIVE: Compute MAS and IRM/CF scores with valid tensor shapes.
# INPUT: Embedding matrices and flattened predictions/labels.
# OUTPUT: MAS losses + score, IRM/CF losses + score.
# REAL-DATA NOTE: Replace sample arrays with real embeddings and prediction/label vectors.

rng = np.random.default_rng(7)

drug_embeddings = rng.normal(size=(16, 64)).astype(np.float32)
prot_embeddings = rng.normal(size=(20, 64)).astype(np.float32)

mas_out = compute_mas_losses(
    drug_embeddings=drug_embeddings,
    prot_embeddings=prot_embeddings,
    mask_ratio=0.15,
    seed=42,
    weight=1.0,
)

print("MAS outputs:")
for k, v in mas_out.items():
    print(f"- {k}: {v}")

n_drugs = 4
n_targets = 5
n_pairs = n_drugs * n_targets
predictions = rng.random(n_pairs).astype(np.float32)
labels = rng.random(n_pairs).astype(np.float32)

irm_cf_out = compute_irm_cf_losses(
    predictions=predictions,
    labels=labels,
    n_drugs=n_drugs,
    n_targets=n_targets,
    n_envs=4,
    pos_threshold=0.5,
    n_cf_pairs=50,
    irm_weight=1.0,
    cf_weight=1.0,
    seed=42,
)

print("\nIRM+CF outputs:")
for k, v in irm_cf_out.items():
    print(f"- {k}: {v}")

MAS outputs:
- mas_drug_loss: 1.4684202706554844e-30
- mas_prot_loss: 2.293951617615872e-30
- l_mas: 3.7623718882713565e-30
- l_mas_weighted: 3.7623718882713565e-30
- mas_score: 1.0

IRM+CF outputs:
- l_irm: 0.005405694129033846
- l_cf: 0.3158836364746094
- env_mses: [0.13696928165346628, 0.1359720946688288, 0.3148890139777226, 0.21904363179447248]
- n_cf_sampled: 11
- irm_cf_score: 0.7568365057054844


## IRM and Counterfactual Breakout (Step-by-Step)

This section explains and tests the Pillar 4 loss helpers in src/c2dti/irm_loss.py:

1. compute_irm_penalty
2. compute_counterfactual_loss

Run cells in order.

In [139]:
import inspect
import numpy as np

from src.c2dti.irm_loss import compute_irm_penalty, compute_counterfactual_loss

# AIM: Confirm the notebook is using the real IRM/CF implementation.
# OBJECTIVE: Import Pillar 4 utilities from source and verify callable signatures.
# INPUT: Source module path and utility symbol names.
# OUTPUT: Printed function signatures.
# REAL-DATA NOTE: Confirms runtime binding to src/c2dti/irm_loss.py.

print("Loaded symbols from src.c2dti.irm_loss")
print("-", compute_irm_penalty.__name__, inspect.signature(compute_irm_penalty))
print("-", compute_counterfactual_loss.__name__, inspect.signature(compute_counterfactual_loss))

Loaded symbols from src.c2dti.irm_loss
- compute_irm_penalty (predictions: 'np.ndarray', labels: 'np.ndarray', drug_indices: 'np.ndarray', target_indices: 'np.ndarray', n_envs: 'int' = 4, seed: 'int' = 42) -> 'Dict[str, float]'
- compute_counterfactual_loss (predictions: 'np.ndarray', labels: 'np.ndarray', drug_indices: 'np.ndarray', target_indices: 'np.ndarray', n_drugs: 'int', n_targets: 'int', pos_threshold: 'float' = 0.5, n_cf_pairs: 'int' = 1000, seed: 'int' = 42) -> 'Dict[str, float]'


In [140]:
# AIM: Verify IRM penalty behavior across drug-split environments.
# OBJECTIVE: Compute per-environment MSE and variance-based IRM penalty.
# INPUT: predictions, labels, drug_indices, target_indices.
# OUTPUT: env_mses, per_env_n, l_irm.
# REAL-DATA NOTE: Replace with flattened predictions/labels from real training batches.

rng = np.random.default_rng(123)

n_drugs = 6
n_targets = 4
n_pairs = n_drugs * n_targets

drug_indices = np.repeat(np.arange(n_drugs), n_targets)
target_indices = np.tile(np.arange(n_targets), n_drugs)

labels = rng.random(n_pairs).astype(np.float32)
predictions = (labels + rng.normal(0, 0.1, n_pairs)).astype(np.float32)

irm_out = compute_irm_penalty(
    predictions=predictions,
    labels=labels,
    drug_indices=drug_indices,
    target_indices=target_indices,
    n_envs=3,
    seed=42,
)

print("IRM outputs:")
for k, v in irm_out.items():
    print(f"- {k}: {v}")

IRM outputs:
- env_mses: [0.011381921960966997, 0.010229957827072061, 0.005510245172121626]
- l_irm: 6.453255247082785e-06
- per_env_n: [8, 8, 8]


In [141]:
# AIM: Verify counterfactual loss behavior on positive-pair target swaps.
# OBJECTIVE: Sample counterfactual pairs and measure mean predicted score on them.
# INPUT: predictions, labels, drug_indices, target_indices, n_drugs, n_targets.
# OUTPUT: l_cf and n_cf_sampled.
# REAL-DATA NOTE: Lower l_cf is better; it indicates fewer shortcut predictions.

# Create a binary-style label view with enough positives for sampling.
labels_binary = (labels > 0.6).astype(np.float32)

cf_out = compute_counterfactual_loss(
    predictions=predictions,
    labels=labels_binary,
    drug_indices=drug_indices,
    target_indices=target_indices,
    n_drugs=n_drugs,
    n_targets=n_targets,
    pos_threshold=0.5,
    n_cf_pairs=100,
    seed=42,
)

print("Counterfactual outputs:")
for k, v in cf_out.items():
    print(f"- {k}: {v}")

Counterfactual outputs:
- l_cf: 0.44700502678751947
- n_cf_sampled: 10


In [142]:
# AIM: Build one combined IRM+CF diagnostic scalar for quick tracking.
# OBJECTIVE: Combine l_irm and l_cf into a bounded score in [0, 1].
# INPUT: irm_out and cf_out from previous cells.
# OUTPUT: combined score and interpretation text.
# REAL-DATA NOTE: Track this score across epochs for stability diagnostics.

irm_weight = 1.0
cf_weight = 1.0
combined = irm_weight * float(irm_out["l_irm"]) + cf_weight * float(cf_out["l_cf"])
irm_cf_score = 1.0 / (1.0 + combined)

print(f"l_irm={irm_out['l_irm']:.6f}")
print(f"l_cf={cf_out['l_cf']:.6f}")
print(f"irm_cf_score={irm_cf_score:.6f}")
print("Interpretation: higher score means better invariance and stronger counterfactual discrimination.")

l_irm=0.000006
l_cf=0.447005
irm_cf_score=0.691080
Interpretation: higher score means better invariance and stronger counterfactual discrimination.


## Unified Scorer Breakout (Step-by-Step)

This section explains and tests the Phase 5 orchestrator in src/c2dti/unified_scorer.py:

1. UnifiedC2DTIScorer import and signature check
2. Full-pillars scoring run (xview + mas + irm + cf)
3. Ablation run (disable some lambdas)

Run cells in order.

In [143]:
import inspect
import numpy as np

from src.c2dti.unified_scorer import UnifiedC2DTIScorer

# AIM: Confirm notebook binding to the real unified scorer implementation.
# OBJECTIVE: Import UnifiedC2DTIScorer and verify constructor/method signatures.
# INPUT: Source class symbol.
# OUTPUT: Printed signatures.
# REAL-DATA NOTE: Confirms runtime is using src/c2dti/unified_scorer.py.

print("Loaded class from src.c2dti.unified_scorer")
print("- __init__", inspect.signature(UnifiedC2DTIScorer.__init__))
print("- score", inspect.signature(UnifiedC2DTIScorer.score))

Loaded class from src.c2dti.unified_scorer
- __init__ (self, cfg: 'Dict[str, Any]') -> 'None'
- score (self, *, predictions: 'np.ndarray', labels: 'np.ndarray', n_drugs: 'int', n_targets: 'int', seq_predictions: 'Optional[np.ndarray]' = None, graph_predictions: 'Optional[np.ndarray]' = None, seq_predictions_pert: 'Optional[np.ndarray]' = None, graph_predictions_pert: 'Optional[np.ndarray]' = None, drug_embeddings: 'Optional[np.ndarray]' = None, prot_embeddings: 'Optional[np.ndarray]' = None) -> 'Dict[str, Any]'


In [144]:
# AIM: Run the unified scorer with all pillars active.
# OBJECTIVE: Compute l_xview, l_mas, l_irm, l_cf, l_total, and unified_causal_score.
# INPUT: predictions/labels, cross-view predictions, and embedding matrices.
# OUTPUT: Unified result dictionary and key scalar values.
# REAL-DATA NOTE: Replace sample arrays with real batch outputs from training.

cfg_full = {
    "lambda_xview": 1.0,
    "lambda_mas": 1.0,
    "lambda_irm": 1.0,
    "lambda_cf": 1.0,
    "cross_view_config": {"weight": 1.0},
    "mas_config": {"mask_ratio": 0.15, "seed": 42, "weight": 1.0},
    "irm_cf_config": {
        "n_envs": 4,
        "pos_threshold": 0.5,
        "n_cf_pairs": 100,
        "irm_weight": 1.0,
        "cf_weight": 1.0,
        "seed": 42,
    },
}

scorer_full = UnifiedC2DTIScorer(cfg_full)

rng = np.random.default_rng(99)
n_drugs = 4
n_targets = 5

pred_matrix = rng.random((n_drugs, n_targets)).astype(np.float32)
label_matrix = rng.random((n_drugs, n_targets)).astype(np.float32)

seq_predictions = rng.random((n_drugs, n_targets)).astype(np.float32)
graph_predictions = rng.random((n_drugs, n_targets)).astype(np.float32)
seq_predictions_pert = rng.random((n_drugs, n_targets)).astype(np.float32)
graph_predictions_pert = rng.random((n_drugs, n_targets)).astype(np.float32)

drug_embeddings = rng.normal(size=(12, 64)).astype(np.float32)
prot_embeddings = rng.normal(size=(18, 64)).astype(np.float32)

unified_full = scorer_full.score(
    predictions=pred_matrix,
    labels=label_matrix,
    n_drugs=n_drugs,
    n_targets=n_targets,
    seq_predictions=seq_predictions,
    graph_predictions=graph_predictions,
    seq_predictions_pert=seq_predictions_pert,
    graph_predictions_pert=graph_predictions_pert,
    drug_embeddings=drug_embeddings,
    prot_embeddings=prot_embeddings,
)

print("Unified full-pillars outputs:")
for key in ["l_xview", "l_mas", "l_irm", "l_cf", "l_total", "unified_causal_score"]:
    print(f"- {key}: {unified_full[key]}")

Unified full-pillars outputs:
- l_xview: 0.4839358627796173
- l_mas: 4.021438912012001e-30
- l_irm: 0.005503772268952899
- l_cf: 0.5240238271653652
- l_total: 1.0134634622139354
- unified_causal_score: 0.49665664104002877


In [145]:
# AIM: Demonstrate ablation behavior by disabling selected pillar lambdas.
# OBJECTIVE: Compare full model score vs ablated configuration score.
# INPUT: Same tensors as full run, but lambdas changed.
# OUTPUT: Side-by-side score comparison and delta.
# REAL-DATA NOTE: Use this exact pattern for your paper ablation matrix.

cfg_ablated = {
    "lambda_xview": 0.0,
    "lambda_mas": 1.0,
    "lambda_irm": 0.0,
    "lambda_cf": 1.0,
    "cross_view_config": {"weight": 1.0},
    "mas_config": {"mask_ratio": 0.15, "seed": 42, "weight": 1.0},
    "irm_cf_config": {
        "n_envs": 4,
        "pos_threshold": 0.5,
        "n_cf_pairs": 100,
        "irm_weight": 1.0,
        "cf_weight": 1.0,
        "seed": 42,
    },
}

scorer_ablated = UnifiedC2DTIScorer(cfg_ablated)
unified_ablated = scorer_ablated.score(
    predictions=pred_matrix,
    labels=label_matrix,
    n_drugs=n_drugs,
    n_targets=n_targets,
    seq_predictions=seq_predictions,
    graph_predictions=graph_predictions,
    seq_predictions_pert=seq_predictions_pert,
    graph_predictions_pert=graph_predictions_pert,
    drug_embeddings=drug_embeddings,
    prot_embeddings=prot_embeddings,
)

print("Ablation comparison:")
print(f"- full unified_causal_score:    {unified_full['unified_causal_score']:.6f}")
print(f"- ablated unified_causal_score: {unified_ablated['unified_causal_score']:.6f}")
print(f"- delta (ablated - full):       {unified_ablated['unified_causal_score'] - unified_full['unified_causal_score']:.6f}")

Ablation comparison:
- full unified_causal_score:    0.496657
- ablated unified_causal_score: 0.656158
- delta (ablated - full):       0.159501


## Phase 6 Evaluation Matrix Breakout (Step-by-Step)

This section explains and tests the Phase 6 evaluation tooling:

1. scripts/run_eval_matrix.py (matrix generation and command sheet)
2. scripts/compile_results.py (detailed and aggregate CSV reports)

Run cells in order.

In [146]:
import subprocess
import sys
from pathlib import Path

# AIM: Confirm notebook binding to the real Phase-6 tooling scripts.
# OBJECTIVE: Verify script file presence and print the exact commands used.
# INPUT: Project root and script paths.
# OUTPUT: Existence checks and runnable command strings.
# REAL-DATA NOTE: Same commands are used for actual evaluation matrix runs.

PROJECT_ROOT = Path.cwd()
run_eval_matrix_script = PROJECT_ROOT / "scripts" / "run_eval_matrix.py"
compile_results_script = PROJECT_ROOT / "scripts" / "compile_results.py"

print("Script checks:")
print("- run_eval_matrix exists:", run_eval_matrix_script.exists(), run_eval_matrix_script)
print("- compile_results exists:", compile_results_script.exists(), compile_results_script)

dry_cmd = [sys.executable, str(run_eval_matrix_script), "--mode", "dry-run"]
compile_cmd = [sys.executable, str(compile_results_script), "--prefix", "C2DTI_EVAL_"]

print("\nCommand templates:")
print("-", " ".join(dry_cmd))
print("-", " ".join(compile_cmd))

Script checks:
- run_eval_matrix exists: True /home/hussen/MINDG/C2DTI/scripts/run_eval_matrix.py
- compile_results exists: True /home/hussen/MINDG/C2DTI/scripts/compile_results.py

Command templates:
- /home/hussen/miniconda3/envs/me-drugban/bin/python /home/hussen/MINDG/C2DTI/scripts/run_eval_matrix.py --mode dry-run
- /home/hussen/miniconda3/envs/me-drugban/bin/python /home/hussen/MINDG/C2DTI/scripts/compile_results.py --prefix C2DTI_EVAL_


In [147]:
# AIM: Generate the Phase-6 command sheet safely.
# OBJECTIVE: Run run_eval_matrix.py in dry-run mode (no training execution).
# INPUT: --mode dry-run
# OUTPUT: Matrix command sheet summary and first few command lines.
# REAL-DATA NOTE: Use --execute only when you are ready to launch real runs.

cmd = [sys.executable, "scripts/run_eval_matrix.py", "--mode", "dry-run"]
res = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)

print("returncode:", res.returncode)
stdout_lines = res.stdout.splitlines()
print("stdout line count:", len(stdout_lines))
print("\n--- dry-run head (first 20 lines) ---")
print("\n".join(stdout_lines[:20]))
if res.returncode != 0:
    print("\n--- stderr ---")
    print(res.stderr)

returncode: 0
stdout line count: 141

--- dry-run head (first 20 lines) ---
# C2DTI Phase-6 Evaluation Matrix
# Mode        : dry-run
# Config dir  : /home/hussen/MINDG/C2DTI/configs/generated_eval_matrix
# Total runs  : 135
001. cd /home/hussen/MINDG/C2DTI && python /home/hussen/MINDG/C2DTI/scripts/run.py --config /home/hussen/MINDG/C2DTI/configs/generated_eval_matrix/davis_random_full_seed10.yaml --dry-run
002. cd /home/hussen/MINDG/C2DTI && python /home/hussen/MINDG/C2DTI/scripts/run.py --config /home/hussen/MINDG/C2DTI/configs/generated_eval_matrix/davis_random_full_seed34.yaml --dry-run
003. cd /home/hussen/MINDG/C2DTI && python /home/hussen/MINDG/C2DTI/scripts/run.py --config /home/hussen/MINDG/C2DTI/configs/generated_eval_matrix/davis_random_full_seed42.yaml --dry-run
004. cd /home/hussen/MINDG/C2DTI && python /home/hussen/MINDG/C2DTI/scripts/run.py --config /home/hussen/MINDG/C2DTI/configs/generated_eval_matrix/davis_random_no_causal_seed10.yaml --dry-run
005. cd /home/hussen/M

In [148]:
# AIM: Compile Phase-6 result tables from generated run summaries.
# OBJECTIVE: Build detailed and aggregate CSV outputs for evaluation matrix reporting.
# INPUT: Prefix filter C2DTI_EVAL_
# OUTPUT: Script status and output/report file listing.
# REAL-DATA NOTE: After real runs, this produces paper-ready summary tables.

cmd = [sys.executable, "scripts/compile_results.py", "--prefix", "C2DTI_EVAL_"]
res_compile = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)

print("returncode:", res_compile.returncode)
print("\n--- compile stdout ---")
print(res_compile.stdout.strip() if res_compile.stdout.strip() else "<no stdout>")
if res_compile.returncode != 0 or res_compile.stderr.strip():
    print("\n--- compile stderr ---")
    print(res_compile.stderr.strip() if res_compile.stderr.strip() else "<no stderr>")

reports_dir = PROJECT_ROOT / "outputs" / "reports"
print("\nReport files:")
for p in sorted(reports_dir.glob("eval_matrix*.csv")):
    print("-", p.name)

returncode: 0

--- compile stdout ---
[INFO] No runs found with prefix=C2DTI_EVAL_

Report files:


## Phase 7 Paper-Facing Docs Breakout (Step-by-Step)

This section checks documentation artifacts tied to publication readiness:

1. docs/C2DTI_4PILLAR_IMPLEMENTATION_ROADMAP.md
2. docs/results_analysis.md
3. docs/paper_draft.md

Run cells in order.

In [149]:
from pathlib import Path

# AIM: Check existence of Phase-7 paper-facing documentation files.
# OBJECTIVE: Verify which artifacts are already present vs missing.
# INPUT: Expected docs file paths.
# OUTPUT: Presence table with absolute paths.
# REAL-DATA NOTE: Missing docs here are non-code blockers for publication packaging.

PROJECT_ROOT = Path.cwd()
docs_expected = [
    PROJECT_ROOT / "docs" / "C2DTI_4PILLAR_IMPLEMENTATION_ROADMAP.md",
    PROJECT_ROOT / "docs" / "results_analysis.md",
    PROJECT_ROOT / "docs" / "paper_draft.md",
]

print("Phase-7 docs status:")
for p in docs_expected:
    print(f"- {p.name:45} exists={p.exists()}  path={p}")

Phase-7 docs status:
- C2DTI_4PILLAR_IMPLEMENTATION_ROADMAP.md       exists=True  path=/home/hussen/MINDG/C2DTI/docs/C2DTI_4PILLAR_IMPLEMENTATION_ROADMAP.md
- results_analysis.md                           exists=False  path=/home/hussen/MINDG/C2DTI/docs/results_analysis.md
- paper_draft.md                                exists=False  path=/home/hussen/MINDG/C2DTI/docs/paper_draft.md


In [150]:
# AIM: Preview the roadmap document content for quick notebook-side audit.
# OBJECTIVE: Display first lines of the main roadmap file.
# INPUT: docs/C2DTI_4PILLAR_IMPLEMENTATION_ROADMAP.md
# OUTPUT: Top section text preview.
# REAL-DATA NOTE: Use this to align implementation and reporting narrative.

roadmap_path = PROJECT_ROOT / "docs" / "C2DTI_4PILLAR_IMPLEMENTATION_ROADMAP.md"

if roadmap_path.exists():
    lines = roadmap_path.read_text(encoding="utf-8").splitlines()
    print("Roadmap preview (first 30 lines):")
    print("\n".join(lines[:30]))
else:
    print("Roadmap file missing:", roadmap_path)

Roadmap preview (first 30 lines):
# C2DTI 4-Pillar Implementation Roadmap → Top Journal Paper

## Architecture Overview

The complete C2DTI model has **4 interdependent pillars**, each enforcing different causal constraints:

```
INPUTS:
  Drug SMILES  ──→ ChemBERTa (frozen) ──→ e_drug ∈ ℝ^768
  Target Seq   ──→ ANKH (frozen)       ──→ e_prot ∈ ℝ^768
  
  Drug Graph   ──→ MixHop GNN          ──→ h_graph ∈ ℝ^d
  Target Graph ──→ MixHop GNN          ──→ h_graph ∈ ℝ^d

OUTPUTS:
  p_seq   = sigmoid(α·e_drug + (1-α)·e_prot) + MLP
  p_graph = sigmoid(h_graph) + MLP
```

---

## Implementation Phases

### Phase 1: Pillar 1 — Dual Pretrained Backbone

**Goal:** Load frozen ChemBERTa + ANKH; create sequence-view encoder.

**Tasks:**

1. **Load pretrained models**


In [152]:
# AIM: Produce a concise publication-readiness docs summary.
# OBJECTIVE: Summarize present/missing files as next writing actions.
# INPUT: docs_expected list from previous cell.
# OUTPUT: Ready/missing counts and specific action items.
# REAL-DATA NOTE: This defines the non-code completion checklist.

present = [p for p in docs_expected if p.exists()]
missing = [p for p in docs_expected if not p.exists()]

print(f"present_docs={len(present)}")
for p in present:
    print(f"  - {p.name}")

print(f"missing_docs={len(missing)}")
for p in missing:
    print(f"  - {p.name}")

if missing:
    print("\nRecommended next writing tasks:")
    for p in missing:
        print(f"- Create {p.name}")
else:
    print("\nAll planned Phase-7 doc artifacts are present.")

present_docs=1
  - C2DTI_4PILLAR_IMPLEMENTATION_ROADMAP.md
missing_docs=2
  - results_analysis.md
  - paper_draft.md

Recommended next writing tasks:
- Create results_analysis.md
- Create paper_draft.md


## Phase 6 Binary Evaluation Matrix Breakout

This section mirrors the regression Phase 6 tooling, but for binary classification.

What it covers:
1. `scripts/run_binary_eval_matrix.py` command generation and pilot/full-run commands.
2. Inspection of the latest binary run summary.
3. `scripts/compile_binary_results.py` report generation.

Run the cells in order.

In [156]:
import subprocess
import sys
from pathlib import Path

# AIM: Confirm notebook binding to the real binary Phase-6 tooling scripts.
# OBJECTIVE: Verify script file presence and print the exact pilot/full-run commands.
# INPUT: Project root and binary script paths.
# OUTPUT: Existence checks and runnable command strings.
# REAL-DATA NOTE: Same commands are used for actual binary evaluation matrix runs.


PROJECT_ROOT = Path.cwd()
run_binary_eval_matrix_script = PROJECT_ROOT / "scripts" / "run_binary_eval_matrix.py"
compile_binary_results_script = PROJECT_ROOT / "scripts" / "compile_binary_results.py"

print("Script checks:")
print("- run_binary_eval_matrix exists:", run_binary_eval_matrix_script.exists(), run_binary_eval_matrix_script)
print("- compile_binary_results exists:", compile_binary_results_script.exists(), compile_binary_results_script)

pilot_cmd = [sys.executable, str(run_binary_eval_matrix_script), "--mode", "run-once", "--execute", "--max-runs", "3"]
full_cmd = [sys.executable, str(run_binary_eval_matrix_script), "--mode", "run-once", "--execute"]
compile_cmd = [sys.executable, str(compile_binary_results_script), "--prefix", "C2DTI_BINARY_EVAL_"]

print("\nCommand templates:")
print("- pilot:", " ".join(pilot_cmd))
print("- full :", " ".join(full_cmd))
print("- compile:", " ".join(compile_cmd))

Script checks:
- run_binary_eval_matrix exists: True /home/hussen/MINDG/C2DTI/scripts/run_binary_eval_matrix.py
- compile_binary_results exists: True /home/hussen/MINDG/C2DTI/scripts/compile_binary_results.py

Command templates:
- pilot: /home/hussen/miniconda3/envs/me-drugban/bin/python /home/hussen/MINDG/C2DTI/scripts/run_binary_eval_matrix.py --mode run-once --execute --max-runs 3
- full : /home/hussen/miniconda3/envs/me-drugban/bin/python /home/hussen/MINDG/C2DTI/scripts/run_binary_eval_matrix.py --mode run-once --execute
- compile: /home/hussen/miniconda3/envs/me-drugban/bin/python /home/hussen/MINDG/C2DTI/scripts/compile_binary_results.py --prefix C2DTI_BINARY_EVAL_


In [154]:
# AIM: Inspect the latest binary pilot run summary and verify it used real data.
# OBJECTIVE: Print the newest summary.json from outputs_binary/runs and highlight key fields.
# INPUT: outputs_binary/runs directory.
# OUTPUT: Latest run path, full summary payload, and quick validation checks.
# REAL-DATA NOTE: dataset_placeholder must be False for a real binary benchmark run.


binary_summary = show_latest_summary(PROJECT_ROOT / "outputs_binary" / "runs")

if binary_summary is not None:
    print("\nQuick checks:")
    print("- status:", binary_summary.get("status"))
    print("- dataset_name:", binary_summary.get("dataset_name"))
    print("- dataset_placeholder:", binary_summary.get("dataset_placeholder"))
    print("- metrics keys:", sorted(binary_summary.get("evaluation_metrics", {}).keys()))

latest_run=/home/hussen/MINDG/C2DTI/outputs_binary/runs/C2DTI_BINARY_EVAL_DAVIS_RANDOM_S42-20260424-204924
{
  "run_name": "C2DTI_BINARY_EVAL_DAVIS_RANDOM_S42",
  "protocol": "P_binary",
  "task_type": "binary_classification",
  "status": "completed",
  "created_at": "2026-04-24T20:49:25",
  "dataset_name": "DAVIS",
  "dataset_placeholder": false,
  "dataset_allow_placeholder": false,
  "num_drugs": 68,
  "num_targets": 379,
  "prediction_path": "/home/hussen/MINDG/C2DTI/outputs_binary/runs/C2DTI_BINARY_EVAL_DAVIS_RANDOM_S42-20260424-204924/predictions.csv",
  "prediction_stats": {
    "mean": 0.9051581025123596,
    "std": 0.0494338795542717,
    "min": 0.5603402256965637,
    "max": 0.9893660545349121
  },
  "binary_threshold": 0.5,
  "evaluation_metrics": {
    "auroc": 0.8693971000041935,
    "auprc": 0.9869009443742691,
    "f1": 0.9640201005025125,
    "accuracy": 0.9305393868839736,
    "sensitivity": 1.0,
    "specificity": 0.0,
    "precision": 0.9305393868839736,
    "thresho

In [155]:
# AIM: Compile binary evaluation reports from generated run summaries.
# OBJECTIVE: Build binary detailed and aggregate CSV outputs for reporting.
# INPUT: Prefix filter C2DTI_BINARY_EVAL_.
# OUTPUT: Script status and output/report file listing.
# REAL-DATA NOTE: After full binary runs, this produces report tables under outputs_binary/reports.


cmd = [sys.executable, "scripts/compile_binary_results.py", "--prefix", "C2DTI_BINARY_EVAL_"]
res_compile_binary = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)

print("returncode:", res_compile_binary.returncode)
print("\n--- compile stdout ---")
print(res_compile_binary.stdout.strip() if res_compile_binary.stdout.strip() else "<no stdout>")
if res_compile_binary.returncode != 0 or res_compile_binary.stderr.strip():
    print("\n--- compile stderr ---")
    print(res_compile_binary.stderr.strip() if res_compile_binary.stderr.strip() else "<no stderr>")

binary_reports_dir = PROJECT_ROOT / "outputs_binary" / "reports"
print("\nReport files:")
for p in sorted(binary_reports_dir.glob("*.csv")):
    print("-", p.name)

returncode: 0

--- compile stdout ---
[OK] Wrote detailed table : /home/hussen/MINDG/C2DTI/outputs_binary/reports/binary_eval_matrix_detailed.csv
[OK] Wrote aggregate table: /home/hussen/MINDG/C2DTI/outputs_binary/reports/binary_eval_matrix_aggregate.csv
[INFO] Included runs      : 3

Report files:
- binary_eval_matrix_aggregate.csv
- binary_eval_matrix_detailed.csv


In [158]:
import csv
from pathlib import Path
from collections import defaultdict

root = Path('outputs/reports')
detailed = root / 'eval_matrix_detailed.csv'
rows = []
with detailed.open('r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows.append(r)

by_name = {}
for r in rows:
    rn = r['run_name']
    if rn not in by_name or r['summary_path'] > by_name[rn]['summary_path']:
        by_name[rn] = r

clean_rows = list(by_name.values())

def seed_num(r):
    try:
        return int(str(r.get('seed', '')).strip())
    except Exception:
        return 0

clean_rows.sort(key=lambda r: (r['dataset'], r['split'], r['ablation'], seed_num(r)))

out_d = root / 'eval_matrix_detailed_dedup.csv'
fields = [
    'run_name','dataset','split','ablation','seed','status','ci','rmse','pearson','spearman','causal_score','l_total','summary_path'
]
with out_d.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(clean_rows)

grp = defaultdict(list)
for r in clean_rows:
    grp[(r['dataset'], r['split'], r['ablation'])].append(r)

def fval(x):
    try:
        return float(x)
    except Exception:
        return float('nan')

def mean(vals):
    vals = [v for v in vals if v == v]
    return (sum(vals)/len(vals)) if vals else None

agg_rows = []
for (dataset, split, ablation), g in sorted(grp.items()):
    agg_rows.append({
        'dataset': dataset,
        'split': split,
        'ablation': ablation,
        'n_runs': len(g),
        'ci_mean': mean([fval(x['ci']) for x in g]),
        'rmse_mean': mean([fval(x['rmse']) for x in g]),
        'pearson_mean': mean([fval(x['pearson']) for x in g]),
        'spearman_mean': mean([fval(x['spearman']) for x in g]),
    })

out_a = root / 'eval_matrix_aggregate_dedup.csv'
fields_a = ['dataset','split','ablation','n_runs','ci_mean','rmse_mean','pearson_mean','spearman_mean']
with out_a.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields_a)
    w.writeheader()
    w.writerows(agg_rows)

print(f'raw_runs={len(rows)}')
print(f'dedup_runs={len(clean_rows)}')
print(f'groups={len(agg_rows)}')
print(f'detailed={out_d}')
print(f'aggregate={out_a}')

raw_runs=138
dedup_runs=135
groups=12
detailed=outputs/reports/eval_matrix_detailed_dedup.csv
aggregate=outputs/reports/eval_matrix_aggregate_dedup.csv
